In [ ]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║  COLAB SETUP — Run this cell FIRST on Google Colab                  ║
# ║  On local Jupyter it is safely skipped.                             ║
# ╚══════════════════════════════════════════════════════════════════════╝
import sys, os

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    print('Google Colab detected — installing packages...')
    os.system('pip install -q biopython diffusers transformers accelerate einops sentencepiece scikit-image')
    print('Packages installed.')

    from google.colab import drive
    drive.mount('/content/drive')

    ZIP_PATH = '/content/drive/MyDrive/lux_project.zip'
    PROJ_DIR = '/content/drive/MyDrive/lux_project'

    os.makedirs(PROJ_DIR, exist_ok=True)

    if not os.path.exists(os.path.join(PROJ_DIR, 'data')):
        print('Unzipping project into Drive...')
        os.system('unzip -q "' + ZIP_PATH + '" -d /content/drive/MyDrive/')
        print('Unzip complete.')
    else:
        print('Project folder already exists at ' + PROJ_DIR)

    os.chdir(PROJ_DIR)
    print('Working directory: ' + os.getcwd())

    # Show GPU info (safe — works on CPU-only runtimes too)
    import shutil
    if shutil.which('nvidia-smi'):
        import subprocess
        gpu = subprocess.run(['nvidia-smi','--query-gpu=name','--format=csv,noheader'],
                             capture_output=True, text=True).stdout.strip()
        print('GPU: ' + gpu)
    else:
        print('GPU: none detected (CPU runtime) — consider switching to GPU in Runtime > Change runtime type')
else:
    print('Running locally — Colab setup skipped.')


In [ ]:
import os
print(os.getcwd())
print(os.listdir('images/'))

# Lux Gene DNA Analysis → Stable Diffusion

**Goal**: Analyse the DNA of bioluminescent lux genes in *Vibrio fischeri*, create a synthetic gene (LuxZ), and generate images with Stable Diffusion driven by real biological features.

**Pipeline overview**:
```
Step 1 → Fetch DNA sequences from NCBI
Step 2 → Extract 3 biological features per gene (GC content, k-mer fingerprint, lux box score)
Step 3 → Build a 3D gene map with PCA
Step 4 → Create LuxZ (synthetic gene placed on the map)
Step 5 → Decode LuxZ back to biological meaning
Step 6 → Convert features into Stable Diffusion prompts
Step 7 → Generate images
```

**Key model choices**:

**ESM2 over k-mers (Step 3b vs Step 3)**: Step 3 encodes each gene as GC content + k-mer frequencies — interpretable and invertible (any PCA coordinate can be projected back to sequence statistics). Step 3b uses ESM2, a protein language model trained on 250M sequences, which captures evolutionary and functional relationships that raw sequence statistics miss. Two genes with similar k-mer profiles may be functionally unrelated; two functionally related genes that diverged in sequence will still cluster in ESM2 space. K-mer PCA is kept for the gene map (axis meanings are readable); ESM2 is used where functional kinship matters (placing LuxZ in Step 4d).

**ControlNet over simple screen blend (Step 8c-2 vs Step 8c)**: Step 8c overlays the backbone trace on the image using screen blend, then runs img2img to unify it. The backbone lines are visible but sit on the surface — SD smooths the blend without reorganising the composition around the protein structure. Step 8c-2 feeds the backbone directly into SD's generation process via ControlNet scribble conditioning, so the protein geometry actively reshapes the image during denoising rather than being received as a pre-blended input.

**Lab blend for the final composite (Step 8d)**: CIE Lab separates luminance (L) from colour (ab). Step 8b carries organic cell depth and bioluminescent colour; Step 8c-2 carries protein geometry but loses the cell atmosphere. A direct RGB blend dilutes both. Working in Lab lets the blend combine L channels from both sources (merging depth and structure) while keeping the ab channels exclusively from 8b — so the bioluminescent colour palette is preserved intact while the protein geometry is incorporated into the luminance layer.

## Setup
Install dependencies if needed:
```bash
pip install biopython scikit-learn matplotlib numpy diffusers transformers accelerate
```

In [ ]:
import os
import json
import subprocess
import numpy as np
import matplotlib.pyplot as plt
from itertools import product
from collections import Counter

from Bio import SeqIO, Entrez
from Bio.SeqUtils import gc_fraction
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

os.makedirs('data/dna', exist_ok=True)
os.makedirs('images', exist_ok=True)

# The 10 lux genes focus on, with their biological role
LUX_GENES = {
    "VF_A0593": {"name": "luxT",  "role": "regulator"},
    "VF_A0918": {"name": "luxG",  "role": "flavin reductase — provides FMNH2 fuel for luciferase"},
    "VF_A0919": {"name": "luxE",  "role": "acyl-protein synthetase — builds aldehyde substrate"},
    "VF_A0920": {"name": "luxB",  "role": "luciferase beta subunit — produces light"},
    "VF_A0921": {"name": "luxA",  "role": "luciferase alpha subunit — produces light"},
    "VF_A0922": {"name": "luxD",  "role": "acyl transferase — builds aldehyde substrate"},
    "VF_A0923": {"name": "luxC",  "role": "acyl reductase — builds aldehyde substrate"},
    "VF_A0924": {"name": "luxI",  "role": "autoinducer synthase — quorum sensing signal"},
    "VF_A0925": {"name": "luxR",  "role": "transcription factor — activates lux operon"},
    "VF_A1058": {"name": "qsrP",  "role": "quorum sensing related"},
}

print('Setup complete.')

---
## Step 1 — Fetch DNA sequences from NCBI

**What this step doing**: Downloading the DNA (nucleotide) sequences for each lux gene from chromosome 2 of *Vibrio fischeri* ES114 (accession `CP000021.2`).

**What we get**:
- `lux_cds.fasta` — the coding DNA sequence of each gene (the gene itself in ATCG)
- `lux_upstream_300bp.fasta` — the 300 bp of DNA *before* each gene starts (the regulatory region)

**Why both?** The coding sequence tells us *what the protein does*. The upstream region tells us *when and how strongly the gene is switched on* — this is where the lux box lives.

In [ ]:
CDS_FILE      = 'data/dna/lux_cds.fasta'
UPSTREAM_FILE = 'data/dna/lux_upstream_300bp.fasta'

if not os.path.exists(CDS_FILE):
    print('DNA files not found. Running fetch_lux_dna.py ...')
    subprocess.run(['python', 'fetch_lux_dna.py'], check=True)
else:
    print('DNA files already exist, skipping download.')

# Load CDS sequences into a dict: locus_tag → SeqRecord
cds_records = {r.id: r for r in SeqIO.parse(CDS_FILE, 'fasta')}

# Load upstream sequences into a dict: locus_tag → SeqRecord
upstream_records = {r.id: r for r in SeqIO.parse(UPSTREAM_FILE, 'fasta')}

print(f'Loaded {len(cds_records)} CDS sequences')
print(f'Loaded {len(upstream_records)} upstream sequences\n')

# Preview
for tag, info in LUX_GENES.items():
    if tag in cds_records:
        seq_len = len(cds_records[tag].seq)
        print(f"{info['name']:6s} ({tag})  {seq_len:>5} bp   {info['role']}")

---
## Step 2a — GC Content

**What it is**: The fraction of bases in a DNA sequence that are either G (guanine) or C (cytosine), rather than A or T.

**Why it matters biologically**:
- G-C pairs form 3 hydrogen bonds vs A-T's 2, making high-GC sequences more stable and harder to unwind
- High GC → gene is likely expressed under stable, controlled conditions
- Low GC → gene may be more flexible, recently acquired, or stress-regulated

**For bioluminescence**: If luxA/B (luciferase) have different GC than luxI/R (quorum sensing), that tells us they evolved or are regulated differently.

In [ ]:
gc_data = {}

for tag, info in LUX_GENES.items():
    if tag not in cds_records:
        continue
    gc = gc_fraction(cds_records[tag].seq)  # returns a value between 0.0 and 1.0
    gc_data[tag] = gc
    print(f"{info['name']:6s}: GC = {gc:.3f} ({gc*100:.1f}%)")

# Plot
fig, ax = plt.subplots(figsize=(9, 4))
names = [LUX_GENES[t]['name'] for t in gc_data]
values = list(gc_data.values())
bars = ax.bar(names, values, color='steelblue')
ax.axhline(0.5, color='red', linestyle='--', linewidth=1, label='50% GC')
ax.set_ylabel('GC Content')
ax.set_title('GC Content per Lux Gene')
ax.set_ylim(0, 1)
ax.legend()
plt.tight_layout()
plt.savefig('images/gc_content.png', dpi=150)
plt.show()
print('\nSaved → images/gc_content.png')

# --- Cache gc_data ---
import json as _json
with open('data/gc_data.json', 'w') as _f: _json.dump(gc_data, _f)
print('gc_data cached -> data/gc_data.json')


---
## Step 2b — k-mer Fingerprint

**What a k-mer is**: A DNA substring of length k. For k=4, every possible 4-letter combination of ACGT (there are 4⁴ = 256 of them: AAAA, AAAC, ... TTTT).

**What we do**: Count how often each of those 256 k-mers appears in a gene's sequence, then normalize by length. This gives a 256-number fingerprint that captures the gene's sequence identity — two genes with similar functions often share similar k-mer profiles.

**Why not just compare sequences directly?** Sequences vary in length. k-mer frequencies normalize for that and are directly usable as a vector of numbers for PCA.

In [ ]:
K = 4
ALL_KMERS = [''.join(p) for p in product('ACGT', repeat=K)]  # 256 possible 4-mers

def kmer_frequencies(seq, k=4):
    """Return a normalized frequency vector over all possible k-mers."""
    seq = str(seq).upper()
    counts = Counter(seq[i:i+k] for i in range(len(seq) - k + 1))
    total = sum(counts.values())
    return np.array([counts.get(km, 0) / total for km in ALL_KMERS])

kmer_data = {}
for tag in gc_data:  # only tags we have CDS for
    kmer_data[tag] = kmer_frequencies(cds_records[tag].seq)

# Show the top 5 most frequent k-mers for luxA as an example
lux_a_tag = 'VF_A0921'
freqs = kmer_data[lux_a_tag]
top5_idx = np.argsort(freqs)[-5:][::-1]
print('Top 5 k-mers in luxA:')
for i in top5_idx:
    print(f'  {ALL_KMERS[i]}: {freqs[i]:.4f}')

print(f'\nEach gene is now a {len(ALL_KMERS)}-dimensional vector.')

---
## Step 2c — Lux Box Score

**What the lux box is**: A specific 20-base DNA sequence (`ACCTGTAGGATCGTACAGGT`)（ref: https://pubmed.ncbi.nlm.nih.gov/2801220/) in the *regulatory region* (upstream) of the lux genes. When enough bacteria are present (quorum sensing), the LuxR protein binds to this exact spot and switches the whole bioluminescence operon ON.

**This is the emission secret**: No lux box binding → no light. Strong lux box → strong, reliable light emission.

**How we score it**: We slide a 20-base window across the upstream sequence and count how many bases match the consensus. Score of 1.0 = perfect match, 0.0 = no resemblance.

In [ ]:
import json as _json, os as _os
if 'gc_data' not in dir() or not gc_data:
    if _os.path.exists('data/gc_data.json'):
        with open('data/gc_data.json') as _f: gc_data = _json.load(_f)
        print('gc_data loaded from cache.')

LUX_BOX_CONSENSUS = 'ACCTGTAGGATCGTACAGGT'  # documented V. fischeri lux box

def lux_box_score(upstream_seq):
    """Scan upstream region and return the best match score against the lux box consensus."""
    seq = str(upstream_seq).upper()
    box_len = len(LUX_BOX_CONSENSUS)
    best = 0.0
    for i in range(len(seq) - box_len + 1):
        window = seq[i:i + box_len]
        score = sum(a == b for a, b in zip(window, LUX_BOX_CONSENSUS)) / box_len
        best = max(best, score)
    return best

luxbox_data = {}
for tag in gc_data:
    if tag in upstream_records:
        score = lux_box_score(upstream_records[tag].seq)
        luxbox_data[tag] = score
        name = LUX_GENES[tag]['name']
        bar = '█' * int(score * 20)
        print(f"{name:6s}: {score:.3f}  {bar}")
    else:
        luxbox_data[tag] = 0.0

print(f'\nHighest score = closest to a real lux box = most strongly regulated by quorum sensing.')

# --- Cache luxbox_data ---
import json as _json
with open('data/luxbox_data.json', 'w') as _f: _json.dump(luxbox_data, _f)
print('luxbox_data cached -> data/luxbox_data.json')


## Step 3 — Build the Gene Map with PCA

**What we have now**: Each gene is described by a vector of 257 numbers — GC content (1 number) + k-mer frequencies (256 numbers). That's impossible to visualise directly.

**What PCA does**: It finds the directions in 257-dimensional space where genes differ the most, then projects everything down to 3 dimensions while preserving as much structure as possible. Think of it as casting a shadow from the angle that loses the least information.

**The result**: Each gene becomes a point in 3D space. Genes with similar sequence composition land close together. This is the gene map — the latent space.

**Key property**: Because the input features are hand-crafted and interpretable, this map is **invertible** — any coordinate can be projected back to recover estimated GC content and dominant k-mer patterns. This is what makes LuxZ decodable in Step 5.


In [ ]:
import json as _json, os as _os
if 'gc_data' not in dir() or not gc_data:
    if _os.path.exists('data/gc_data.json'):
        with open('data/gc_data.json') as _f: gc_data = _json.load(_f)
        print('gc_data loaded from cache.')
if 'luxbox_data' not in dir() or not luxbox_data:
    if _os.path.exists('data/luxbox_data.json'):
        with open('data/luxbox_data.json') as _f: luxbox_data = _json.load(_f)
        print('luxbox_data loaded from cache.')

tags_ordered = [t for t in LUX_GENES if t in gc_data and t in kmer_data]
names_ordered = [LUX_GENES[t]['name'] for t in tags_ordered]

# Build feature matrix: each row = one gene
gc_col   = np.array([[gc_data[t]] for t in tags_ordered])
kmer_mat = np.array([kmer_data[t] for t in tags_ordered])
X = np.hstack([gc_col, kmer_mat])  # shape: (n_genes, 257)

# Standardize so no single feature dominates due to scale
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# PCA to 3D
pca = PCA(n_components=3)
coords = pca.fit_transform(X_scaled)  # shape: (n_genes, 3)

explained = pca.explained_variance_ratio_ * 100
print(f'PCA explains {explained[0]:.1f}% + {explained[1]:.1f}% + {explained[2]:.1f}% = {sum(explained):.1f}% of total variation')

# Colour by lux box score (brighter = higher score)
lux_scores = [luxbox_data.get(t, 0) for t in tags_ordered]

fig = plt.figure(figsize=(10, 7))
ax = fig.add_subplot(111, projection='3d')
sc = ax.scatter(coords[:,0], coords[:,1], coords[:,2],
                c=lux_scores, cmap='plasma', s=120)

for i, name in enumerate(names_ordered):
    ax.text(coords[i,0], coords[i,1], coords[i,2] + 0.05, name, fontsize=9)

plt.colorbar(sc, ax=ax, label='Lux Box Score (quorum sensing strength)')
ax.set_xlabel(f'PC1 ({explained[0]:.1f}%)')
ax.set_ylabel(f'PC2 ({explained[1]:.1f}%)')
ax.set_zlabel(f'PC3 ({explained[2]:.1f}%)')
ax.set_title('Lux Gene Map (DNA-based, 3D PCA)')
plt.tight_layout()
plt.savefig('images/gene_map_3d.png', dpi=150)
plt.show()
print('Saved → images/gene_map_3d.png')

# Save coordinates for reference
gene_coords = {tags_ordered[i]: {'name': names_ordered[i], 'pca': coords[i].tolist()} for i in range(len(tags_ordered))}
with open('data/gene_map.json', 'w') as f:
    json.dump(gene_coords, f, indent=2)

---
## Step 3b — ESM2 Protein Embeddings 

**What changes**: Instead of hand-crafted k-mer frequencies, we use **ESM2** — a protein language model trained on 250M protein sequences. Each gene gets a 320-dimensional embedding that captures evolutionary and functional meaning far deeper than k-mers.

**Pipeline**:
```
CDS DNA  →  translate to protein (BioPython)  →  ESM2 embedding (320D)  →  PCA 3D  →  new gene map
```

**Result**: `coords` and `gene_coords` are overwritten with ESM2-based positions. LuxZ will be placed more biologically meaningfully.

**Key property**: This map is more accurate but **not invertible** — ESM2's dimensions carry no human-readable labels. Proximity means functional kinship, but the axes cannot be decoded. This is why Step 3 is kept alongside Step 3b rather than replaced by it.


In [ ]:
import json as _json, os as _os
if 'luxbox_data' not in dir() or not luxbox_data:
    if _os.path.exists('data/luxbox_data.json'):
        with open('data/luxbox_data.json') as _f: luxbox_data = _json.load(_f)
        print('luxbox_data loaded from cache.')

from transformers import EsmModel, EsmTokenizer
from Bio.Seq import Seq
import torch, json, numpy as np, matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

# Preserve k-mer PCA so Step 5 decode still runs without errors
kmer_pca, kmer_scaler = pca, scaler

# --- 1. Translate CDS DNA -> protein ---
protein_seqs = {}
for tag, info in LUX_GENES.items():
    if tag in cds_records:
        protein = str(Seq(str(cds_records[tag].seq)).translate(to_stop=True))
        protein_seqs[tag] = protein
        print(f"{info['name']:6s}: {len(protein)} aa")

# --- 2. Load ESM2 (8M params, fast on MPS) ---
print('\nLoading ESM2...')
esm_tokenizer = EsmTokenizer.from_pretrained('facebook/esm2_t6_8M_UR50D')
esm_model     = EsmModel.from_pretrained('facebook/esm2_t6_8M_UR50D')
esm_model.eval()
esm_device = 'cuda' if torch.cuda.is_available() else ('mps' if torch.backends.mps.is_available() else 'cpu')
esm_model  = esm_model.to(esm_device)
print(f'ESM2 loaded on {esm_device}')

# --- 3. Get embeddings (mean pool over sequence tokens) ---
embeddings = {}
with torch.no_grad():
    for tag, seq in protein_seqs.items():
        inputs = esm_tokenizer(seq, return_tensors='pt', add_special_tokens=True).to(esm_device)
        out    = esm_model(**inputs)
        emb    = out.last_hidden_state[0, 1:-1].mean(0).cpu().numpy()  # exclude [CLS]/[EOS]
        embeddings[tag] = emb
print(f'Embeddings: {len(embeddings)} genes x {emb.shape[0]} dims')

# --- 4. PCA to 3D ---
tags_ordered  = [t for t in LUX_GENES if t in embeddings]
names_ordered = [LUX_GENES[t]['name'] for t in tags_ordered]

X_esm    = np.array([embeddings[t] for t in tags_ordered])
scaler   = StandardScaler()
X_scaled = scaler.fit_transform(X_esm)
pca      = PCA(n_components=3)
coords   = pca.fit_transform(X_scaled)
explained = pca.explained_variance_ratio_ * 100
print(f'ESM2 PCA: {explained[0]:.1f}% + {explained[1]:.1f}% + {explained[2]:.1f}% = {sum(explained):.1f}%')

# --- 5. Update gene_coords (same format as Step 3) ---
lux_scores  = [luxbox_data.get(t, 0) for t in tags_ordered]
gene_coords = {tags_ordered[i]: {'name': names_ordered[i], 'pca': coords[i].tolist()} for i in range(len(tags_ordered))}
with open('data/gene_map.json', 'w') as f:
    json.dump(gene_coords, f, indent=2)

# --- 6. Plot new map ---
fig = plt.figure(figsize=(10, 7))
ax  = fig.add_subplot(111, projection='3d')
sc  = ax.scatter(coords[:,0], coords[:,1], coords[:,2], c=lux_scores, cmap='plasma', s=120)
for i, name in enumerate(names_ordered):
    ax.text(coords[i,0], coords[i,1], coords[i,2]+0.05, name, fontsize=9)
plt.colorbar(sc, ax=ax, label='Lux Box Score (quorum sensing strength)')
ax.set_xlabel(f'PC1 ({explained[0]:.1f}%)')
ax.set_ylabel(f'PC2 ({explained[1]:.1f}%)')
ax.set_zlabel(f'PC3 ({explained[2]:.1f}%)')
ax.set_title('Lux Gene Map — ESM2 Protein Embeddings')
plt.tight_layout()
plt.savefig('images/gene_map_esm2.png', dpi=150)
plt.show()
print('Saved -> images/gene_map_esm2.png')
print('\ncoords + gene_coords updated -> re-run Steps 4-10 for improved LuxZ + animation')


# --- Cache protein_seqs ---
import json as _json
with open('data/protein_seqs.json', 'w') as _f: _json.dump(protein_seqs, _f)
print('protein_seqs cached -> data/protein_seqs.json')

# --- Cache pca + scaler (needed by Step 5 decode) ---
import pickle as _pkl
with open('data/pca.pkl', 'wb') as _f: _pkl.dump(pca, _f)
with open('data/scaler.pkl', 'wb') as _f: _pkl.dump(scaler, _f)
print('pca + scaler cached -> data/pca.pkl, data/scaler.pkl')


In [ ]:
import json as _json, os as _os
if 'gc_data' not in dir() or not gc_data:
    if _os.path.exists('data/gc_data.json'):
        with open('data/gc_data.json') as _f: gc_data = _json.load(_f)
        print('gc_data loaded from cache.')
if 'luxbox_data' not in dir() or not luxbox_data:
    if _os.path.exists('data/luxbox_data.json'):
        with open('data/luxbox_data.json') as _f: luxbox_data = _json.load(_f)
        print('luxbox_data loaded from cache.')

# --- Compare k-mer PCA vs ESM2 PCA side by side ---
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

# Recompute old k-mer coords for comparison
tags = [t for t in LUX_GENES if t in gc_data and t in kmer_data]
names = [LUX_GENES[t]['name'] for t in tags]
gc_col   = np.array([[gc_data[t]] for t in tags])
kmer_mat = np.array([kmer_data[t] for t in tags])
X_kmer   = np.hstack([gc_col, kmer_mat])
X_kmer_s = kmer_scaler.fit_transform(X_kmer)
old_coords = kmer_pca.fit_transform(X_kmer_s)

# Current coords are ESM2-based
esm_coords = coords

# Plot side by side (PC1 vs PC2 only for clarity)
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
lux_scores_list = [luxbox_data.get(t, 0) for t in tags]

for ax, c, title in zip(axes,
                         [old_coords, esm_coords],
                         ['k-mer + GC (Step 3)', 'ESM2 protein embeddings (Step 3b)']):
    sc = ax.scatter(c[:,0], c[:,1], c=lux_scores_list, cmap='plasma', s=150, zorder=3)
    for i, name in enumerate(names):
        ax.annotate(name, (c[i,0], c[i,1]),
                    textcoords='offset points', xytext=(6, 4),
                    fontsize=9, color='white')
    ax.set_xlabel('PC1', color='white')
    ax.set_ylabel('PC2', color='white')
    ax.set_title(title, color='white')
    ax.set_facecolor('#111')
    ax.tick_params(colors='white')
    for spine in ax.spines.values():
        spine.set_edgecolor('#555')
    fig.colorbar(sc, ax=ax, label='Lux Box Score').ax.yaxis.label.set_color('white')

fig.patch.set_facecolor('#111')
plt.suptitle('Gene Map Comparison — k-mer vs ESM2', color='white', fontsize=12)
plt.tight_layout()
plt.savefig('images/gene_map_comparison.png', dpi=150, facecolor='#111')
plt.show()
print('Saved -> images/gene_map_comparison.png')


---
## Step 3c — ESMFold: Predict 3D Protein Structures

**What ESMFold does**: Given a protein sequence, it predicts the full 3D folded structure — where every atom sits in space. No experimental data needed.

**Why this matters**: Each lux protein has a different shape. Instead of generating images purely from text prompts, we can feed the *actual predicted protein geometry* into ControlNet as a conditioning image

**Output per gene**:
- `.pdb` file — full 3D structure
- `_structure.png` — 2D backbone trace (CA atoms, coloured N→C) — used as ControlNet input

**Note**: ESMFold is ~2.7 GB. First run will download and cache it. ~30–60s per protein on MPS.


In [ ]:
import json as _json, os as _os
if 'protein_seqs' not in dir() or not protein_seqs:
    if _os.path.exists('data/protein_seqs.json'):
        with open('data/protein_seqs.json') as _f: protein_seqs = _json.load(_f)
        print('protein_seqs loaded from cache.')

from transformers import EsmForProteinFolding, EsmTokenizer
from transformers.models.esm.openfold_utils.protein import to_pdb, Protein as OFProtein
from transformers.models.esm.openfold_utils.feats import atom14_to_atom37
import torch, os, numpy as np, matplotlib.pyplot as plt
from sklearn.decomposition import PCA as skPCA
from Bio.Seq import Seq

os.makedirs('data/structures', exist_ok=True)
os.makedirs('images/structures', exist_ok=True)

# Translate CDS -> protein (reuse if protein_seqs already defined from Step 3b)
if 'protein_seqs' not in dir():
    protein_seqs = {}
    for tag, info in LUX_GENES.items():
        if tag in cds_records:
            protein_seqs[tag] = str(Seq(str(cds_records[tag].seq)).translate(to_stop=True))

def render_backbone(positions, mask, name, save_path):
    ca_mask   = mask[:, 1].astype(bool)
    ca_coords = positions[:, 1][ca_mask]
    if len(ca_coords) < 3:
        return None
    pca2d = skPCA(n_components=2)
    xy    = pca2d.fit_transform(ca_coords)
    colors = plt.cm.cool(np.linspace(0, 1, len(xy)))
    fig, ax = plt.subplots(figsize=(4, 4), facecolor='black')
    ax.set_facecolor('black')
    ax.plot(xy[:,0], xy[:,1], color='#334455', linewidth=1.2, zorder=1)
    ax.scatter(xy[:,0], xy[:,1], c=colors, s=12, zorder=2, linewidths=0)
    ax.set_aspect('equal')
    ax.axis('off')
    ax.set_title(name, color='white', fontsize=10, pad=4)
    plt.tight_layout(pad=0.2)
    plt.savefig(save_path, dpi=128, bbox_inches='tight', facecolor='black')
    plt.close()
    return save_path

def render_backbone_from_pdb(pdb_path, name, save_path):
    """Render backbone directly from a saved PDB file — no model needed."""
    ca_xyz = []
    with open(pdb_path) as f:
        for line in f:
            if line.startswith('ATOM') and line[12:16].strip() == 'CA':
                x, y, z = float(line[30:38]), float(line[38:46]), float(line[46:54])
                ca_xyz.append([x, y, z])
    if len(ca_xyz) < 3:
        return None
    ca_xyz = np.array(ca_xyz)
    pca2d  = skPCA(n_components=2)
    xy     = pca2d.fit_transform(ca_xyz)
    colors = plt.cm.cool(np.linspace(0, 1, len(xy)))
    fig, ax = plt.subplots(figsize=(4, 4), facecolor='black')
    ax.set_facecolor('black')
    ax.plot(xy[:,0], xy[:,1], color='#334455', linewidth=1.2, zorder=1)
    ax.scatter(xy[:,0], xy[:,1], c=colors, s=12, zorder=2, linewidths=0)
    ax.set_aspect('equal')
    ax.axis('off')
    ax.set_title(name, color='white', fontsize=10, pad=4)
    plt.tight_layout(pad=0.2)
    plt.savefig(save_path, dpi=128, bbox_inches='tight', facecolor='black')
    plt.close()
    return save_path

# --- Check which genes still need folding ---
needs_folding = {}
for tag, info in LUX_GENES.items():
    if tag not in protein_seqs:
        continue
    name = info['name']
    pdb_path = f'data/structures/{name}.pdb'
    if not os.path.exists(pdb_path):
        needs_folding[tag] = info

structure_images = {}

# --- Render images from existing PDBs (no model needed) ---
for tag, info in LUX_GENES.items():
    if tag not in protein_seqs:
        continue
    name     = info['name']
    pdb_path = f'data/structures/{name}.pdb'
    img_path = f'images/structures/{name}_structure.png'
    if os.path.exists(pdb_path):
        render_backbone_from_pdb(pdb_path, name, img_path)
        structure_images[name] = img_path
        print(f'  {name}: rendered from existing PDB')

# --- Only load ESMFold if some genes are missing PDBs ---
if needs_folding:
    print(f'\nFolding {len(needs_folding)} missing genes: {[v["name"] for v in needs_folding.values()]}')
    fold_device = 'cuda' if torch.cuda.is_available() else 'cpu'
    fold_model  = EsmForProteinFolding.from_pretrained('facebook/esmfold_v1', low_cpu_mem_usage=True)
    fold_model  = fold_model.to(fold_device)
    if fold_device == 'cuda':
        fold_model = fold_model.half()
        fold_model.esm = fold_model.esm.float()
    fold_model.eval()
    print(f'ESMFold loaded on {fold_device}.')

    with torch.no_grad():
        for tag, info in needs_folding.items():
            name = info['name']
            seq  = protein_seqs[tag]
            print(f'  Folding {name} ({len(seq)} aa)...')
            inputs = fold_tokenizer([seq], return_tensors='pt', add_special_tokens=False)
            inputs = {k: v.to(fold_device) for k, v in inputs.items()}
            output = fold_model(**inputs)
            positions = atom14_to_atom37(output['positions'][-1], output)
            positions = positions.squeeze(0).cpu().numpy()
            mask      = output['atom37_atom_exists'].squeeze(0).cpu().numpy()
            pdb_path  = f'data/structures/{name}.pdb'
            pdbs = to_pdb(OFProtein(
                aatype        = output['aatype'].squeeze(0).cpu().numpy(),
                atom_positions= positions,
                atom_mask     = mask,
                residue_index = output['residue_index'].squeeze(0).cpu().numpy() + 1,
                b_factors     = np.zeros_like(mask),
                chain_index   = np.zeros(mask.shape[0], dtype=int),
            ))
            with open(pdb_path, 'w') as f:
                f.write(pdbs)
            img_path = f'images/structures/{name}_structure.png'
            render_backbone(positions, mask, name, img_path)
            structure_images[name] = img_path
            del inputs, output, positions, mask
            if fold_device == 'cuda':
                torch.cuda.empty_cache()
            print(f'  Saved -> {pdb_path}')

    del fold_model
    if fold_device == 'cuda':
        torch.cuda.empty_cache()
else:
    print('All PDB files already exist — ESMFold not loaded.')

# --- Display grid ---
names_done = list(structure_images.keys())
cols = 4
rows = (len(names_done) + cols - 1) // cols
fig, axes = plt.subplots(rows, cols, figsize=(cols*3, rows*3), facecolor='black')
axes = axes.flatten()
from PIL import Image as PILImage
for i, name in enumerate(names_done):
    img = PILImage.open(structure_images[name])
    axes[i].imshow(img)
    axes[i].axis('off')
for j in range(len(names_done), len(axes)):
    axes[j].axis('off')
plt.suptitle('Predicted Protein Structures — Lux Genes (ESMFold)', color='white', fontsize=12)
plt.tight_layout()
plt.savefig('images/structures/all_structures.png', dpi=150, facecolor='black')
plt.show()
print('Saved -> images/structures/all_structures.png')
print('\nstructure_images dict ready for Step 7b.')


---
## Step 4 — Create LuxZ (Synthetic Gene)

**The concept**: On our gene map, genes that produce light (luxA, luxB) cluster in one area, and genes that control *when* to produce light (luxI, luxR) cluster in another. LuxZ is a hypothetical gene that sits between these two clusters — something that both *senses the environment* and *contributes to emission*.

**The math**: LuxZ = 70% toward the light-production cluster + 30% toward the quorum sensing cluster. This is not random — it reflects a biological hypothesis that LuxZ would be a hybrid regulator-emitter.

**This is what makes it a digital life form**: We're inventing a gene that nature hasn't made, placing it inside a real biological coordinate system.

In [ ]:
import json as _json, os as _os
if 'gene_coords' not in dir() or not gene_coords:
    if _os.path.exists('data/gene_map.json'):
        with open('data/gene_map.json') as _f: gene_coords = _json.load(_f)
        print('gene_coords loaded from gene_map.json.')

def get_coord(gene_name):
    for tag, info in gene_coords.items():
        if info['name'] == gene_name:
            return np.array(info['pca'])
    raise ValueError(f'{gene_name} not found in gene map')

# Light-production cluster: luxA and luxB (the two subunits of luciferase)
light_center = (get_coord('luxA') + get_coord('luxB')) / 2

# Quorum sensing cluster: luxI (makes the signal) and luxR (receives the signal)
qs_center = (get_coord('luxI') + get_coord('luxR')) / 2

# LuxZ: weighted position between the two clusters
LIGHT_WEIGHT = 0.7
QS_WEIGHT    = 0.3
luxZ_pos = LIGHT_WEIGHT * light_center + QS_WEIGHT * qs_center

print(f'Light cluster center: {light_center.round(3)}')
print(f'QS cluster center:    {qs_center.round(3)}')
print(f'LuxZ position:        {luxZ_pos.round(3)}')

# Visualise: gene map + LuxZ
fig = plt.figure(figsize=(10, 7))
ax = fig.add_subplot(111, projection='3d')
ax.scatter(coords[:,0], coords[:,1], coords[:,2], c=lux_scores, cmap='plasma', s=100, alpha=0.8)

for i, name in enumerate(names_ordered):
    ax.text(coords[i,0], coords[i,1], coords[i,2] + 0.05, name, fontsize=8)

# Plot LuxZ as a star marker
ax.scatter(*luxZ_pos, c='lime', s=250, marker='*', zorder=5, label='LuxZ (synthetic)')
ax.text(*luxZ_pos + 0.05, 'LuxZ', fontsize=10, color='lime', fontweight='bold')

# Draw lines from LuxZ to its parents
for center, label in [(light_center, 'luxA/B'), (qs_center, 'luxI/R')]:
    ax.plot([luxZ_pos[0], center[0]], [luxZ_pos[1], center[1]], [luxZ_pos[2], center[2]],
            'lime', linestyle='--', alpha=0.4)

ax.set_title('Gene Map + LuxZ (synthetic gene)')
ax.legend()
plt.tight_layout()
plt.savefig('images/gene_map_luxZ.png', dpi=150)
plt.show()
print('Saved → images/gene_map_luxZ.png')
# --- Cache luxZ position + weights ---
import json as _json, numpy as _np
with open('data/luxZ_position.json', 'w') as _f:
    _json.dump({'luxZ_pos': luxZ_pos.tolist(), 'LIGHT_WEIGHT': LIGHT_WEIGHT, 'QS_WEIGHT': QS_WEIGHT}, _f)
print('luxZ_pos cached -> data/luxZ_position.json')


---
## Step 4b — LuxZ's Real Identity: Chimeric Protein Sequence

**What we know so far**: LuxZ was placed at a *coordinate* in PCA/ESM2 space (70% toward luxA/B, 30% toward luxI/R). But it has never had a real amino-acid sequence.

**What we do now**:
1. Align the **luxA** and **luxI** protein sequences with BioPython
2. At each aligned position:
   - If both residues are present: pick luxA with **p=0.70** or luxI with **p=0.30** (fixed seed=42)
   - If only one parent has a residue: keep it unconditionally
3. Save the chimeric protein sequence → this is LuxZ's *actual* molecular identity

**Biological meaning**: luxA encodes the α-subunit of luciferase (light-emitting enzyme). luxI encodes the autoinducer synthase (quorum-sensing signal generator). A chimera of the two would be a protein that *both emits light and senses its environment* — perfectly suited to reframe as a **water / environment sensor** for the physical installation.


In [ ]:
import json as _json, os as _os
if 'protein_seqs' not in dir() or not protein_seqs:
    if _os.path.exists('data/protein_seqs.json'):
        with open('data/protein_seqs.json') as _f: protein_seqs = _json.load(_f)
        print('protein_seqs loaded from cache.')

import os, json, numpy as np
from Bio.Seq import Seq
from Bio import pairwise2
from Bio.pairwise2 import format_alignment

# ── 1. Load DNA and translate ────────────────────────────────────────────
from Bio import SeqIO
cds_dir = 'data/dna'
cds_records = {}
for fn in os.listdir(cds_dir):
    if fn.endswith('.fasta'):
        for rec in SeqIO.parse(os.path.join(cds_dir, fn), 'fasta'):
            cds_records[rec.id] = rec

# Also check main CDS file
CDS_FILE = 'data/dna/lux_cds.fasta'
if os.path.exists(CDS_FILE):
    for rec in SeqIO.parse(CDS_FILE, 'fasta'):
        tag = rec.id.split('_')[0].lower()  # e.g. 'luxA'
        for candidate in ['luxA','luxB','luxI','luxR','luxC','luxD','luxE']:
            if candidate.lower() in rec.id.lower() or candidate.lower() in rec.description.lower():
                cds_records[candidate] = rec

# Build protein_seqs from LUX_GENES dict (already in memory if cell 14 ran)
try:
    _ = protein_seqs
    print('Using protein_seqs from memory')
except NameError:
    protein_seqs = {}
    LUX_GENES_TAGS = ['luxA','luxB','luxI','luxR','luxC','luxD','luxE','luxG','luxT','qsrP']
    for tag in LUX_GENES_TAGS:
        if tag in cds_records:
            protein_seqs[tag] = str(Seq(str(cds_records[tag].seq)).translate(to_stop=True))
    print('Rebuilt protein_seqs from DNA files')

print('Available proteins:', list(protein_seqs.keys()))

# ── 2. Build name→sequence mapping (protein_seqs keys are locus tags) ──
name_to_seq = {}
for tag, seq in protein_seqs.items():
    if tag in LUX_GENES:
        name_to_seq[LUX_GENES[tag]['name']] = seq
    else:
        name_to_seq[tag] = seq  # fallback if already named
print('name_to_seq keys:', list(name_to_seq.keys()))

# ── 3. Align luxA (70%) and luxI (30%) ───────────────────────────────────
seqA = name_to_seq.get('luxA', '')
seqI = name_to_seq.get('luxI', '')

if not seqA or not seqI:
    raise ValueError(f'luxA or luxI missing. Available: {list(name_to_seq.keys())}')

print(f'luxA: {len(seqA)} aa | luxI: {len(seqI)} aa')

# Global alignment, match=2, mismatch=-1, gap_open=-0.5, gap_extend=-0.1
alns = pairwise2.align.globalms(seqA, seqI, 2, -1, -0.5, -0.1)
best = alns[0]
aligned_A, aligned_I = best.seqA, best.seqB
print(f'Alignment length: {len(aligned_A)}')
print(format_alignment(*best)[:300])  # preview first 300 chars

# ── 3. Build chimera: at each position pick A (p=0.70) or I (p=0.30) ────
rng = np.random.default_rng(seed=42)
chimera_aa = []
source_log = []  # track which parent each residue came from

for aa_A, aa_I in zip(aligned_A, aligned_I):
    gap_A = (aa_A == '-')
    gap_I = (aa_I == '-')
    if gap_A and gap_I:
        continue  # double gap, skip
    elif gap_A:
        # Only luxI contributes here
        # if rng.random() < 0.30:
            chimera_aa.append(aa_I)
            source_log.append('I')
    elif gap_I:
        # Only luxA contributes here
        # if rng.random() < 0.70:
            chimera_aa.append(aa_A)
            source_log.append('A')
    else:
        # Both present — weighted pick
        if rng.random() < 0.70:
            chimera_aa.append(aa_A)
            source_log.append('A')
        else:
            chimera_aa.append(aa_I)
            source_log.append('I')

luxZ_chimera_seq = ''.join(chimera_aa)
from_A = source_log.count('A')
from_I = source_log.count('I')
print(f'\nLuxZ chimera: {len(luxZ_chimera_seq)} aa')
print(f'  from luxA: {from_A} ({from_A/len(source_log)*100:.1f}%)')
print(f'  from luxI: {from_I} ({from_I/len(source_log)*100:.1f}%)')
print(f'\nSequence (first 80 aa):')
print(luxZ_chimera_seq[:80])

# ── 4. Save chimeric sequence ─────────────────────────────────────────────
os.makedirs('data', exist_ok=True)
with open('data/luxZ_chimera.fasta', 'w') as f:
    f.write('>luxZ_chimera LuxZ synthetic chimera (70% luxA + 30% luxI)\n')
    for i in range(0, len(luxZ_chimera_seq), 60):
        f.write(luxZ_chimera_seq[i:i+60] + '\n')
print('Saved -> data/luxZ_chimera.fasta')


---
## Step 4c — Fold Real LuxZ with ESMFold

Now that LuxZ has a real sequence, we predict its 3D structure using ESMFold.  
This replaces the blended backbone image from earlier — LuxZ now has a *genuine* predicted structure.  

**Memory note**: ESMFold runs in fp16 on CUDA to keep GPU memory under 8 GB. 
If the kernel crashes, ensure you are on a GPU runtime (T4 or better).


In [ ]:
import os, torch, numpy as np, matplotlib.pyplot as plt
from transformers import EsmForProteinFolding, EsmTokenizer
from transformers.models.esm.openfold_utils.protein import to_pdb, Protein as OFProtein

OUTPUT_PDB   = 'data/structures/luxZ.pdb'
OUTPUT_IMAGE = 'images/structures/luxZ_structure.png'
os.makedirs('data/structures', exist_ok=True)
os.makedirs('images/structures', exist_ok=True)

# Load chimeric sequence
try:
    _ = luxZ_chimera_seq
except NameError:
    with open('data/luxZ_chimera.fasta') as f:
        lines = [l.strip() for l in f if not l.startswith('>')]
    luxZ_chimera_seq = ''.join(lines)

# Skip folding if PDB already exists
if os.path.exists(OUTPUT_PDB):
    print(f'PDB already exists, loading from {OUTPUT_PDB}')
    with open(OUTPUT_PDB) as f:
        pdb_str = f.read()
else:
    print(f'Folding LuxZ chimera: {len(luxZ_chimera_seq)} aa')
    fold_device = 'cuda' if torch.cuda.is_available() else ('mps' if torch.backends.mps.is_available() else 'cpu')
    print(f'Loading ESMFold on {fold_device}...')
    fold_tokenizer = EsmTokenizer.from_pretrained('facebook/esmfold_v1')
    fold_model     = EsmForProteinFolding.from_pretrained('facebook/esmfold_v1', low_cpu_mem_usage=True)
    fold_model     = fold_model.to(fold_device)
    if fold_device == 'cuda':
        fold_model = fold_model.half()
        fold_model.esm = fold_model.esm.float()
    fold_model.eval()
    with torch.no_grad():
        inputs = fold_tokenizer([luxZ_chimera_seq], return_tensors='pt', add_special_tokens=False)
        inputs = {k: v.to(fold_device) for k, v in inputs.items()}
        output = fold_model(**inputs)
    pdb_str = fold_model.output_to_pdb(output)[0]
    with open(OUTPUT_PDB, 'w') as f:
        f.write(pdb_str)
    print(f'Saved PDB -> {OUTPUT_PDB}')
    del fold_model, inputs, output
    if torch.backends.mps.is_available(): torch.mps.empty_cache()

# Parse pLDDT from B-factor column in PDB
plddt_list = []
for line in pdb_str.split('\n'):
    if line.startswith('ATOM') and line[12:16].strip() == 'CA':
        plddt_list.append(float(line[60:66]))
plddt = np.array(plddt_list)
print(f'Mean pLDDT: {plddt.mean():.1f} (>70 = confident, >90 = very confident)')

# Parse Cα coordinates from PDB for visualisation
ca_xyz = []
for line in pdb_str.split('\n'):
    if line.startswith('ATOM') and line[12:16].strip() == 'CA':
        x, y, z = float(line[30:38]), float(line[38:46]), float(line[46:54])
        ca_xyz.append([x, y, z])
ca_xyz = np.array(ca_xyz)

# 3D backbone trace — clean black background, no colorbar (matches other gene structure images)
fig = plt.figure(figsize=(6, 6), facecolor='black')
ax  = fig.add_subplot(111, projection='3d')
ax.set_facecolor('black')
ax.plot(ca_xyz[:,0], ca_xyz[:,1], ca_xyz[:,2], lw=0.8, color='#444444', alpha=0.8)
ax.scatter(ca_xyz[:,0], ca_xyz[:,1], ca_xyz[:,2],
           c=plddt[:len(ca_xyz)], cmap='cool', s=12, zorder=5)
ax.set_title('luxZ_chimera', color='white', fontsize=10)
ax.set_axis_off()
ax.grid(False)
plt.tight_layout(pad=0)
plt.savefig(OUTPUT_IMAGE, dpi=150, bbox_inches='tight', facecolor='black')
plt.show()
print(f'Saved -> {OUTPUT_IMAGE}')

print('Step 4c done')


---
## Step 4d — LuxZ's Nearest Biological Neighbour

We now embed the chimeric LuxZ sequence with **ESM2** and compute cosine similarity against:
- All 10 existing lux operon genes
- A small curated set of **water / environment sensing proteins** (osmosensors, hydrophilins)

The nearest neighbour tells us what is LuxZ biologically closest to?*  
This grounds the artistic reframing of LuxZ as a water-detecting gene.


In [ ]:
import json as _json, os as _os
if 'protein_seqs' not in dir() or not protein_seqs:
    if _os.path.exists('data/protein_seqs.json'):
        with open('data/protein_seqs.json') as _f: protein_seqs = _json.load(_f)
        print('protein_seqs loaded from cache.')
import json as _json, os as _os
if 'gene_coords' not in dir() or not gene_coords:
    if _os.path.exists('data/gene_map.json'):
        with open('data/gene_map.json') as _f: gene_coords = _json.load(_f)
        print('gene_coords loaded from gene_map.json.')

import os, json, numpy as np, torch, matplotlib.pyplot as plt
from transformers import EsmModel, EsmTokenizer
from sklearn.metrics.pairwise import cosine_similarity

# ── 1. Load ESM2 ─────────────────────────────────────────────────────────
esm2_device = 'mps' if torch.backends.mps.is_available() else ('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Loading ESM2 on {esm2_device}...')
esm2_tok   = EsmTokenizer.from_pretrained('facebook/esm2_t6_8M_UR50D')
esm2_model = EsmModel.from_pretrained('facebook/esm2_t6_8M_UR50D')
esm2_model.eval().to(esm2_device)
print('ESM2 ready')

def embed_sequence(seq, tok, model, device):
    with torch.no_grad():
        inp = tok(seq, return_tensors='pt', add_special_tokens=True).to(device)
        out = model(**inp)
        return out.last_hidden_state[0, 1:-1].mean(0).cpu().numpy()

# ── 2. Embed LuxZ chimera ──────────────────────────────────────────────────
try:
    _ = luxZ_chimera_seq
except NameError:
    with open('data/luxZ_chimera.fasta') as f:
        luxZ_chimera_seq = ''.join(l.strip() for l in f if not l.startswith('>'))

luxZ_emb = embed_sequence(luxZ_chimera_seq, esm2_tok, esm2_model, esm2_device)
print(f'LuxZ chimera embedding: {luxZ_emb.shape[0]} dims')

# ── 3. Embed all lux genes ────────────────────────────────────────────────
from Bio import SeqIO
from Bio.Seq import Seq

try:
    _ = protein_seqs
except NameError:
    protein_seqs = {}
    for rec in SeqIO.parse('data/dna/lux_cds.fasta', 'fasta'):
        for tag in ['luxA','luxB','luxI','luxR','luxC','luxD','luxE','luxG','luxT','qsrP']:
            if tag.lower() in rec.id.lower() or tag.lower() in rec.description.lower():
                protein_seqs[tag] = str(Seq(str(rec.seq)).translate(to_stop=True))

lux_embeddings = {}
for tag, seq in protein_seqs.items():
    lux_embeddings[tag] = embed_sequence(seq, esm2_tok, esm2_model, esm2_device)
    print(f'  embedded {tag}')

# ── 4. Curated water / environment sensor sequences ───────────────────────
WATER_SENSORS = {
    'EnvZ_osmosensor':
        'MQDFLSHLRTEAQRSLQEQLEQIRQQLQDLTPEVQEAVQRFQEQIRDLHQKISDLHETLKMAAP',
    'OmpR_regulator':
        'MKILIVDDHPMLREGILEYLLQEEGYQVVEATDGAEALEYLEQDPDLILLDIMLPGMDGLELCREIRSDSATPIIMLTAKDDE',
    'KdpE_waterK':
        'MKILIVDDHAILREGILEYLLEEEGYEVVEATDGEAALRYLQQDPDLIVLDIMLPGIDGVELCREIRSDSSTPIIMLTAKDDE',
    'Dehydrin_LEA':
        'MSDQHKKEDPAAMDSSKHQEKKGEMDSSKHEEKPSGMDSSKHQEKKGEMDSSKHEEKPSGMDSSKHEEK',
}

sensor_embeddings = {}
for name, seq in WATER_SENSORS.items():
    sensor_embeddings[name] = embed_sequence(seq, esm2_tok, esm2_model, esm2_device)
    print(f'  embedded {name}')

# ── 5. Compute cosine similarity ─────────────────────────────────────────
all_embs  = {**lux_embeddings, **sensor_embeddings}
all_names = list(all_embs.keys())
X         = np.array([all_embs[n] for n in all_names])
sims      = cosine_similarity([luxZ_emb], X)[0]

ranked = sorted(zip(all_names, sims), key=lambda x: -x[1])
print('\nLuxZ chimera — nearest neighbours (cosine similarity):')
print(f'{"Name":<25} {"Similarity":>12}')
print('-' * 38)
for name, sim in ranked:
    marker = ' ← NEAREST' if name == ranked[0][0] else (
             ' ← closest water sensor' if name in WATER_SENSORS and name == min(
                 [(n,s) for n,s in ranked if n in WATER_SENSORS], key=lambda x:-x[1])[0] else '')
    print(f'{name:<25} {sim:>12.4f}{marker}')

nearest_name, nearest_sim = ranked[0]
sensor_ranked = [(n,s) for n,s in ranked if n in WATER_SENSORS]
closest_sensor, sensor_sim = sensor_ranked[0]

# ── 6. Visualise similarity bar chart (white background) ─────────────────
fig, ax = plt.subplots(figsize=(9, 5), facecolor='white')
ax.set_facecolor('white')
colors = ['#00c0ff' if n in WATER_SENSORS else '#ffd700' for n in all_names]
ax.barh(all_names, sims, color=colors)
ax.axvline(nearest_sim, color='black', linestyle='--', linewidth=1, alpha=0.5)
ax.set_xlabel('Cosine Similarity to LuxZ chimera', color='black')
ax.set_title('LuxZ Chimera — Nearest Neighbours in ESM2 Space', fontsize=12, color='black')
ax.tick_params(colors='black')
for spine in ax.spines.values():
    spine.set_color('black')
from matplotlib.patches import Patch
ax.legend(handles=[Patch(color='#ffd700', label='lux operon gene'),
                   Patch(color='#00c0ff', label='water / env sensor')],
          loc='lower right', framealpha=0.8)
plt.tight_layout()
plt.savefig('images/luxZ_nearest_neighbours.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()
print('Saved -> images/luxZ_nearest_neighbours.png')

# ── 7. Update LuxZ SD prompt for water-sensor identity ───────────────────
print(f'\nNearest biological neighbour: {nearest_name} (sim={nearest_sim:.4f})')
print(f'Closest water sensor: {closest_sensor} (sim={sensor_sim:.4f})')

LUXZ_WATER_PROMPT = (
    'bioluminescent chimeric protein floating in water, '
    'translucent aqueous membrane sensing water molecules, '
    'glowing cyan and blue filaments submerged in liquid, '
    'osmotic signal transduction structure, '
    'microscopic water-responsive molecular antenna, '
    'deep ocean bioluminescence, cinematic light diffraction, '
    'photorealistic scientific visualisation'
)

with open('data/sd_prompts.json') as f:
    prompts = json.load(f)
prompts['luxZ_water'] = LUXZ_WATER_PROMPT
with open('data/sd_prompts.json', 'w') as f:
    json.dump(prompts, f, indent=2)
print('\nLuxZ water-sensor prompt saved to data/sd_prompts.json as "luxZ_water"')
print(f'\nPrompt:\n  {LUXZ_WATER_PROMPT}')

del esm2_model
if esm2_device == 'mps':
    torch.mps.empty_cache()
print('\nESM2 memory freed')

## Step 5 — Decode LuxZ: Reading Back Its Biological Properties

**What we do**: LuxZ exists as a 3D coordinate in PCA space. We reverse the compression back into the original 257-dimensional feature space — recovering its implied GC content, dominant k-mer profile, and lux box score.

**Why this matters**:
- These decoded properties feed directly into Step 6 to build LuxZ's Stable Diffusion prompt
- It places LuxZ on equal footing with the real lux genes,describing in the same biological language

**Note**: The reverse projection is an approximation since PCA is lossy. The lux box score is estimated as a weighted average of its parent genes (luxB and luxI).

In [ ]:
import pickle as _pkl, json as _json, os as _os, numpy as _np
if 'pca' not in dir():
    if _os.path.exists('data/pca.pkl'):
        with open('data/pca.pkl','rb') as _f: pca = _pkl.load(_f)
        with open('data/scaler.pkl','rb') as _f: scaler = _pkl.load(_f)
        print('pca + scaler loaded from cache.')
    else:
        raise RuntimeError('pca not found — run Cell 15 (Step 3b) first.')
if 'luxZ_pos' not in dir():
    if _os.path.exists('data/luxZ_position.json'):
        with open('data/luxZ_position.json') as _f: _d = _json.load(_f)
        luxZ_pos = _np.array(_d['luxZ_pos'])
        LIGHT_WEIGHT, QS_WEIGHT = _d['LIGHT_WEIGHT'], _d['QS_WEIGHT']
        print('luxZ_pos + weights loaded from cache.')
    else:
        raise RuntimeError('luxZ_pos not found — run Cell 20 (Step 4) first.')
import json as _json, os as _os
if 'gc_data' not in dir() or not gc_data:
    if _os.path.exists('data/gc_data.json'):
        with open('data/gc_data.json') as _f: gc_data = _json.load(_f)
        print('gc_data loaded from cache.')
if 'luxbox_data' not in dir() or not luxbox_data:
    if _os.path.exists('data/luxbox_data.json'):
        with open('data/luxbox_data.json') as _f: luxbox_data = _json.load(_f)
        print('luxbox_data loaded from cache.')

# Reverse PCA: 3D position → 257-dimensional feature vector
luxZ_scaled = pca.inverse_transform(luxZ_pos.reshape(1, -1))
luxZ_features = scaler.inverse_transform(luxZ_scaled)[0]

luxZ_gc = LIGHT_WEIGHT * gc_data.get('VF_A0925', 0) + QS_WEIGHT * gc_data.get('VF_A0924', 0)
luxZ_kmers   = luxZ_features[1:]  # 256 k-mer frequencies

# Top 5 k-mers that define LuxZ's sequence identity
top5_idx = np.argsort(luxZ_kmers)[-5:][::-1]

print(f'LuxZ decoded properties:')
print(f'  GC content:  {luxZ_gc:.3f} ({luxZ_gc*100:.1f}%)')
print(f'  Top 5 characteristic k-mers:')
for i in top5_idx:
    print(f'    {ALL_KMERS[i]}: {luxZ_kmers[i]:.4f}')

# Lux box score: LuxZ inherits a weighted average from its parents
luxZ_lux_box = LIGHT_WEIGHT * luxbox_data.get('VF_A0921', 0) + QS_WEIGHT * luxbox_data.get('VF_A0924', 0)
print(f'  Estimated lux box score: {luxZ_lux_box:.3f}')

# Save all LuxZ properties
luxZ_properties = {
    'position_3d': luxZ_pos.tolist(),
    'gc_content':  luxZ_gc,
    'lux_box_score': luxZ_lux_box,
    'top_kmers': [ALL_KMERS[i] for i in top5_idx],
    'role': 'synthetic hybrid — bridges light emission and quorum sensing'
}
with open('data/luxZ_dna_properties.json', 'w') as f:
    json.dump(luxZ_properties, f, indent=2)
print('\nSaved → data/luxZ_dna_properties.json')

---
## Step 6 — Convert Biological Features into Stable Diffusion Prompts

**The mapping logic**:

| Biological feature | What it means | → Prompt element |
|---|---|---|
| lux box score | How strongly quorum sensing activates this gene | light intensity (dormant → blazing) |
| GC content | Sequence stability | colour temperature (cold blue = stable, warm amber = flexible) |
| Role (light vs QS) | What the gene does | visual role in scene |


Each gene gets its own prompt. Including LuxZ.

In [ ]:
import json as _json, os as _os
if 'gc_data' not in dir() or not gc_data:
    if _os.path.exists('data/gc_data.json'):
        with open('data/gc_data.json') as _f: gc_data = _json.load(_f)
        print('gc_data loaded from cache.')
if 'luxbox_data' not in dir() or not luxbox_data:
    if _os.path.exists('data/luxbox_data.json'):
        with open('data/luxbox_data.json') as _f: luxbox_data = _json.load(_f)
        print('luxbox_data loaded from cache.')

INTENSITY_WORDS = ['dormant', 'faint', 'softly glowing', 'radiant', 'blazing']

def gc_to_color(gc):
    if gc > 0.36:
        return 'cold electric blue-white'
    elif gc > 0.33:
        return 'blue-green bioluminescent'
    else:
        return 'warm amber-green bioluminescent'

def make_prompt(name, gc, lux_score, role):
    intensity_idx = min(int(lux_score * 5), 4)
    intensity = INTENSITY_WORDS[intensity_idx]
    color     = gc_to_color(gc)
    return (
        f"microscopic view of a single bacterial cell named {name}, "
        f"{intensity} {color} light, {role}, "
        f"dark deep ocean background, scientific macro photography, "
        f"bioluminescent organism, photorealistic, cinematic lighting, 8k"
    )

prompts = {}

# Prompts for real genes
for tag in tags_ordered:
    info   = LUX_GENES[tag]
    name   = info['name']
    gc     = gc_data[tag]
    score  = luxbox_data.get(tag, 0)
    role   = info['role']
    prompts[name] = make_prompt(name, gc, score, role)

# Prompt for LuxZ (synthetic)
prompts['luxZ'] = make_prompt('LuxZ', luxZ_gc, luxZ_lux_box,
    'synthetic hybrid gene bridging light emission and quorum sensing')

# Display all prompts
print('Generated prompts:\n')
for gene, prompt in prompts.items():
    print(f'[{gene}]')
    print(f'  {prompt}\n')

with open('data/sd_prompts.json', 'w') as f:
    json.dump(prompts, f, indent=2)
print('Saved → data/sd_prompts.json')

---
## Step 7 — Generate Images with Stable Diffusion

**What happens here**: We feed each gene's prompt into Stable Diffusion and it generates an image. Each image is a visual interpretation of that gene's biological properties — not random, but grounded in real DNA analysis.

**Device detection**:
- Apple Silicon Mac (M1/M2/M3/M4) → uses `mps` (Metal GPU)
- NVIDIA GPU → uses `cuda`  
- Otherwise → uses `cpu` (slow but works)

**First run**: The model (~4 GB) will download automatically and cache locally.

In [ ]:
import json
with open('data/sd_prompts.json') as f:
    prompts = json.load(f)
print(list(prompts.keys()))


In [ ]:
import torch
from diffusers import StableDiffusionPipeline

# Pick the best available device
if torch.cuda.is_available():
    device = 'cuda'
    dtype  = torch.float16
elif torch.backends.mps.is_available():
    device = 'mps'
    dtype  = torch.float32
else:
    device = 'cpu'
    dtype  = torch.float32

print(f'Using device: {device}')

pipe = StableDiffusionPipeline.from_pretrained(
    'runwayml/stable-diffusion-v1-5',
    torch_dtype=dtype,
    safety_checker=None
)
pipe = pipe.to(device)
print('Model loaded.')

In [ ]:
# Generate one image per gene
# Each image uses a fixed seed so results are reproducible
SEED            = 42
INFERENCE_STEPS = 30  # lower = faster, higher = more detail
IMAGE_SIZE      = 512

generated_images = {}

for gene, prompt in prompts.items():
    print(f'Generating [{gene}] ...')
    generator = torch.Generator(device=device).manual_seed(SEED)
    result = pipe(
        prompt,
        height=IMAGE_SIZE,
        width=IMAGE_SIZE,
        num_inference_steps=INFERENCE_STEPS,
        generator=generator
    )
    img = result.images[0]
    path = f'images/sd_{gene}.png'
    img.save(path)
    generated_images[gene] = img
    print(f'  Saved → {path}')

print('\nAll images generated.')

In [ ]:
# Display all generated images in a grid
n = len(generated_images)
cols = 4
rows = (n + cols - 1) // cols

fig, axes = plt.subplots(rows, cols, figsize=(cols * 4, rows * 4))
axes = axes.flatten()

for i, (gene, img) in enumerate(generated_images.items()):
    axes[i].imshow(img)
    axes[i].set_title(gene, fontsize=11,
                      color='lime' if gene == 'luxZ' else 'white')
    axes[i].axis('off')

for j in range(i + 1, len(axes)):
    axes[j].axis('off')

fig.patch.set_facecolor('black')
plt.suptitle('Lux Genes as Digital Life Forms — Stable Diffusion', color='white', fontsize=14)
plt.tight_layout()
plt.savefig('images/sd_all_genes.png', dpi=150, facecolor='black')
plt.show()
print('Saved → images/sd_all_genes.png')

---
## Step 7b — Protein Structure + SD Image Composite

**What this does**:
1. Loads each gene's ESMFold backbone trace from Step 3c (LuxZ uses its chimeric structure from Step 4c)
2. Screen-blends the backbone onto its Step 7 SD image — the protein structure appears glowing inside the bioluminescent cell
3. Passes each composite through img2img (strength=0.4) to unify the style into a coherent scene


In [ ]:
import os, numpy as np, torch, matplotlib.pyplot as plt
from PIL import Image, ImageFilter
from diffusers import StableDiffusionImg2ImgPipeline

# --- Helper: screen blend ---
def screen_blend(base, layer, weight=1.0):
    a = np.array(base).astype(float) / 255
    b = np.array(layer).astype(float) / 255 * weight
    return Image.fromarray((np.clip(1-(1-a)*(1-b), 0, 1)*255).astype(np.uint8))

# --- Helper: load + boost backbone image ---
def load_backbone(name, size=512):
    path = f'images/structures/{name}_structure.png'
    if not os.path.exists(path):
        return Image.new('RGB', (size, size), 0)
    img = Image.open(path).convert('RGB').resize((size, size))
    arr = np.clip(np.array(img).astype(float) * 3.0, 0, 255).astype(np.uint8)
    return Image.fromarray(arr)

# --- Step 1: Screen-blend backbone onto Step 7 SD image ---
# LuxZ uses its real ESMFold structure from Step 4c (luxZ_structure.png)
os.makedirs('images/composite', exist_ok=True)

all_genes  = list(generated_images.keys())   # from Step 7
backbones  = {g: load_backbone(g) for g in all_genes}

composites = {}
for gene, sd_img in generated_images.items():
    bb   = backbones.get(gene)
    if bb is None:
        composites[gene] = sd_img
        continue
    comp = screen_blend(sd_img.resize((512,512)), bb, weight=0.85)
    comp.save(f'images/composite/{gene}_composite.png')
    composites[gene] = comp

print(f'Composites saved -> images/composite/')

# Preview composites
n = len(composites)
cols = 4
rows = (n + cols - 1) // cols
fig, axes = plt.subplots(rows, cols, figsize=(cols*4, rows*4), facecolor='black')
axes = axes.flatten()
for i, (gene, img) in enumerate(composites.items()):
    axes[i].imshow(img)
    axes[i].set_title(gene, fontsize=10, color='lime' if gene == 'luxZ' else 'white')
    axes[i].axis('off')
for j in range(i+1, len(axes)):
    axes[j].axis('off')
plt.suptitle('Step 7b — Backbone + SD Composite (before img2img)', color='white', fontsize=13)
plt.tight_layout()
plt.savefig('images/composite/all_composites_raw.png', dpi=150, facecolor='black')
plt.show()
print('Saved -> images/composite/all_composites_raw.png')

# --- Step 3: Pass composites through img2img to unify style ---
if torch.cuda.is_available():
    device, dtype = 'cuda', torch.float16
elif torch.backends.mps.is_available():
    device, dtype = 'mps', torch.float32
else:
    device, dtype = 'cpu', torch.float32

img2img_pipe = StableDiffusionImg2ImgPipeline.from_pretrained(
    'runwayml/stable-diffusion-v1-5',
    torch_dtype=dtype,
    safety_checker=None
).to(device)
print('img2img pipeline loaded.')

# Load prompts (from Step 6)
import json as _json
with open('data/sd_prompts.json') as f:
    prompts = _json.load(f)

unified = {}
for gene, comp_img in composites.items():
    prompt = prompts.get(gene, prompts.get('luxZ', ''))
    print(f'  Refining [{gene}]...')
    generator = torch.Generator(device=device).manual_seed(42)
    result = img2img_pipe(
        prompt=prompt,
        image=comp_img,
        strength=0.4,        # keep composite structure, let SD unify style
        num_inference_steps=25,
        generator=generator
    ).images[0]
    result.save(f'images/composite/{gene}_unified.png')
    unified[gene] = result

print('All unified images saved.')

# Final grid
fig, axes = plt.subplots(rows, cols, figsize=(cols*4, rows*4), facecolor='black')
axes = axes.flatten()
for i, (gene, img) in enumerate(unified.items()):
    axes[i].imshow(img)
    axes[i].set_title(gene, fontsize=10, color='lime' if gene == 'luxZ' else 'white')
    axes[i].axis('off')
for j in range(i+1, len(axes)):
    axes[j].axis('off')
plt.suptitle('Step 7b — Backbone + SD Composite (after img2img unification)', color='white', fontsize=13)
plt.tight_layout()
plt.savefig('images/composite/all_composites_unified.png', dpi=150, facecolor='black')
plt.show()
print('Saved -> images/composite/all_composites_unified.png')


In [ ]:
import torch, os, json, numpy as np, matplotlib.pyplot as plt
from PIL import Image
from diffusers import StableDiffusionImg2ImgPipeline

# --- Tweak this to control how much SD changes the composite ---
STRENGTH = 0.65   # 0.5 = keeps more structure, 0.8 = more SD creativity

if torch.cuda.is_available():
    device, dtype = 'cuda', torch.float16
elif torch.backends.mps.is_available():
    device, dtype = 'mps', torch.float32
else:
    device, dtype = 'cpu', torch.float32

pipe = StableDiffusionImg2ImgPipeline.from_pretrained(
    'runwayml/stable-diffusion-v1-5',
    torch_dtype=dtype,
    safety_checker=None
).to(device)

with open('data/sd_prompts.json') as f:
    prompts = json.load(f)

os.makedirs('images/composite', exist_ok=True)
refined = {}

for gene, prompt in prompts.items():
    comp_path = f'images/composite/{gene}_composite.png'
    if not os.path.exists(comp_path):
        print(f'  [{gene}] composite not found, skipping')
        continue
    print(f'  Refining [{gene}] strength={STRENGTH}...')
    comp_img  = Image.open(comp_path).convert('RGB').resize((512, 512))
    generator = torch.Generator(device=device).manual_seed(42)
    result    = pipe(
        prompt=prompt,
        negative_prompt='blurry, flat, matte, white background, cartoon, dull',
        image=comp_img,
        strength=STRENGTH,
        num_inference_steps=30,
        guidance_scale=8.0,
        generator=generator
    ).images[0]
    out_path = f'images/composite/{gene}_refined.png'
    result.save(out_path)
    refined[gene] = result

print(f'Saved {len(refined)} refined images.')

# Display grid
n    = len(refined)
cols = 4
rows = (n + cols - 1) // cols
fig, axes = plt.subplots(rows, cols, figsize=(cols*4, rows*4), facecolor='black')
axes = axes.flatten()
for i, (gene, img) in enumerate(refined.items()):
    axes[i].imshow(img)
    axes[i].set_title(gene, fontsize=10, color='lime' if gene == 'luxZ' else 'white')
    axes[i].axis('off')
for j in range(i+1, len(axes)):
    axes[j].axis('off')
plt.suptitle(f'Backbone + SD Refined (strength={STRENGTH})', color='white', fontsize=13)
plt.tight_layout()
plt.savefig(f'images/composite/all_refined_s{int(STRENGTH*10)}.png', dpi=150, facecolor='black')
plt.show()
print(f'Saved -> images/composite/all_refined_s{int(STRENGTH*10)}.png')


---
## Step 8 — img2img: Apply Biological Style onto your TouchDesigner Visual

**What this does**: Instead of generating from a blank canvas, SD starts from your TD screenshot and transforms it using each gene's biological prompt. Your TD visual becomes the structure — SD applies the bioluminescence biology on top.

**The `strength` parameter** controls how much SD changes your image:
- `0.3` → subtle stylization, TD structure stays very visible
- `0.6` → balanced — keeps the composition, applies bioluminescence style
- `0.9` → heavy transformation, TD image is just a rough guide

Try different values and see which feels right for your aesthetic.

In [ ]:
import json
import torch
import matplotlib.pyplot as plt
from PIL import Image
from diffusers import StableDiffusionImg2ImgPipeline

# Load prompts from saved file — no need to re-run Steps 2-6
with open('data/sd_prompts.json') as f:
    prompts = json.load(f)

# Load TD image
td_image = Image.open('TDimage1.tif').convert('RGB').resize((512, 512))
print(f'TD image loaded: {td_image.size}')
plt.imshow(td_image)
plt.title('Your TouchDesigner input')
plt.axis('off')
plt.show()

# Pick device
if torch.cuda.is_available():
    device, dtype = 'cuda', torch.float16
elif torch.backends.mps.is_available():
    device, dtype = 'mps', torch.float32
else:
    device, dtype = 'cpu', torch.float32
print(f'Using device: {device}')

# Load img2img pipeline — reuses cached model, no re-download
img2img_pipe = StableDiffusionImg2ImgPipeline.from_pretrained(
    'runwayml/stable-diffusion-v1-5',
    torch_dtype=dtype,
    safety_checker=None
).to(device)

STRENGTH        = 0.6  # 0.3 = keep TD look, 0.6 = balanced, 0.9 = almost ignore TD
SEED            = 42
INFERENCE_STEPS = 30

img2img_images = {}

for gene, prompt in prompts.items():
    print(f'Generating [{gene}] ...')
    generator = torch.Generator(device=device).manual_seed(SEED)
    result = img2img_pipe(
        prompt=prompt,
        image=td_image,
        strength=STRENGTH,
        num_inference_steps=INFERENCE_STEPS,
        generator=generator
    )
    img = result.images[0]
    path = f'images/img2img_{gene}.png'
    img.save(path)
    img2img_images[gene] = img
    print(f'  Saved -> {path}')

print('All img2img images generated.')

# Display grid
n    = len(img2img_images)
cols = 4
rows = (n + cols - 1) // cols
fig, axes = plt.subplots(rows, cols, figsize=(cols * 4, rows * 4))
axes = axes.flatten()

for i, (gene, img) in enumerate(img2img_images.items()):
    axes[i].imshow(img)
    axes[i].set_title(gene, fontsize=11, color='lime' if gene == 'luxZ' else 'white')
    axes[i].axis('off')

for j in range(i + 1, len(axes)):
    axes[j].axis('off')

fig.patch.set_facecolor('black')
plt.suptitle(f'TD Visual x Lux Biology (strength={STRENGTH})', color='white', fontsize=14)
plt.tight_layout()
plt.savefig('images/img2img_all_genes.png', dpi=150, facecolor='black')
plt.show()
print('Saved -> images/img2img_all_genes.png')

## Step 8b — TD Input × Protein Structure Composite

**What this does**: Combines three sources into one image per gene:
1. **TD image** — your cell cluster composition, providing the visual structure
2. **Step 7b refined** — the protein backbone baked into a bioluminescent scene
3. **Biological prompt** — used by img2img to unify the final style

**How the blend works**:
- Linear blend: 50% TD + 50% Step 7b refined
- Screen blend TD on top again to preserve its bright structural elements
- img2img at strength=0.6 unifies everything into one coherent scene

**Result**: The TD cell cluster composition stays recognisable, while the protein structure and bioluminescent style from Step 7b are layered into it.

In [ ]:
import torch, os, json, numpy as np, matplotlib.pyplot as plt
from PIL import Image
from diffusers import StableDiffusionImg2ImgPipeline

# --- How much TD input vs refined composite to blend before img2img ---
TD_WEIGHT      = 0.5    # 0.0 = only refined composite, 1.0 = only TD
STRENGTH       = 0.6    # how much SD changes the blend

if torch.cuda.is_available():
    device, dtype = 'cuda', torch.float16
elif torch.backends.mps.is_available():
    device, dtype = 'mps', torch.float32
else:
    device, dtype = 'cpu', torch.float32

pipe = StableDiffusionImg2ImgPipeline.from_pretrained(
    'runwayml/stable-diffusion-v1-5',
    torch_dtype=dtype,
    safety_checker=None
).to(device)

with open('data/sd_prompts.json') as f:
    prompts = json.load(f)

# --- Load TD image ---
td_image = Image.open('TDimage1.tif').convert('RGB').resize((512, 512))

os.makedirs('images/td_composite', exist_ok=True)
results = {}

def screen_blend_imgs(base, layer):
    a = np.array(base).astype(float) / 255
    b = np.array(layer).astype(float) / 255
    return Image.fromarray((np.clip(1-(1-a)*(1-b), 0, 1)*255).astype(np.uint8))

for gene, prompt in prompts.items():
    # Load Step 7b refined image — fall back to Step 7 SD image if not found
    refined_path = f'images/composite/{gene}_refined.png'
    if not os.path.exists(refined_path):
        refined_path = f'images/sd_{gene}.png'
    if not os.path.exists(refined_path):
        print(f'  [{gene}] no source image found, skipping')
        continue

    refined = Image.open(refined_path).convert('RGB').resize((512, 512))

    # Blend TD + refined composite
    td_arr      = np.array(td_image).astype(float) / 255
    refined_arr = np.array(refined).astype(float) / 255
    blended_arr = TD_WEIGHT * td_arr + (1 - TD_WEIGHT) * refined_arr
    blended     = Image.fromarray((np.clip(blended_arr, 0, 1)*255).astype(np.uint8))

    # Screen blend the TD on top to preserve its bright elements
    blended = screen_blend_imgs(blended, td_image)

    # Pass through img2img
    print(f'  [{gene}] TD_WEIGHT={TD_WEIGHT} strength={STRENGTH}...')
    generator = torch.Generator(device=device).manual_seed(42)
    result    = pipe(
        prompt=prompt,
        negative_prompt='blurry, flat, matte, white background, cartoon, dull',
        image=blended,
        strength=STRENGTH,
        num_inference_steps=30,
        guidance_scale=8.0,
        generator=generator
    ).images[0]

    out_path = f'images/td_composite/{gene}_td_composite.png'
    result.save(out_path)
    results[gene] = result

print(f'\nSaved {len(results)} images -> images/td_composite/')

# --- Display grid ---
n    = len(results)
cols = 4
rows = (n + cols - 1) // cols
fig, axes = plt.subplots(rows, cols, figsize=(cols*4, rows*4), facecolor='black')
axes = axes.flatten()
for i, (gene, img) in enumerate(results.items()):
    axes[i].imshow(img)
    axes[i].set_title(gene, fontsize=10, color='lime' if gene == 'luxZ' else 'white')
    axes[i].axis('off')
for j in range(i+1, len(axes)):
    axes[j].axis('off')
fig.patch.set_facecolor('black')
plt.suptitle(f'TD × Protein Structure (TD={TD_WEIGHT}, strength={STRENGTH})', color='white', fontsize=13)
plt.tight_layout()
plt.savefig('images/td_composite/all_td_composite.png', dpi=150, facecolor='black')
plt.show()
print('Saved -> images/td_composite/all_td_composite.png')


--- 
## Step 8c — Screen Blend Attempt (superseded by Step 8c-2)

**What this does**: Takes the Step 8 img2img output and the ESMFold backbone trace, screen-blends them together, then passes the result through img2img at strength=0.45 to unify the layers.

**Why it didn't work**: The backbone appears as glowing lines overlaid on the composition, but they sit on the surface without influencing the underlying scene. SD's img2img pass at low strength smooths the blend without reorganising the image around the protein geometry — the structure is visible but cosmetically applied rather than structurally integrated. The result is a backbone glow floating on top of the TD composition, not reshaping it.

This led directly to the ControlNet approach in Step 8c-2, where backbone geometry is injected into the SD generation process itself rather than pre-blended into the input image.

**Output**: `images/8c/{gene}_8c.png` (not used in downstream steps)

In [ ]:
# Backbone fallback: genes that share a structure with another gene
BB_FALLBACK = {'luxZ_water': 'luxZ'}

import torch, os, json, numpy as np, matplotlib.pyplot as plt
from PIL import Image
from diffusers import StableDiffusionImg2ImgPipeline

os.makedirs('images/8c', exist_ok=True)

with open('data/sd_prompts.json') as f:
    prompts = json.load(f)

def screen_blend(base, layer, weight=1.0):
    a = np.array(base).astype(float) / 255
    b = np.array(layer).astype(float) / 255 * weight
    return Image.fromarray((np.clip(1-(1-a)*(1-b), 0, 1)*255).astype(np.uint8))

if torch.cuda.is_available():
    device, dtype = 'cuda', torch.float16
elif torch.backends.mps.is_available():
    device, dtype = 'mps', torch.float32
else:
    device, dtype = 'cpu', torch.float32

pipe = StableDiffusionImg2ImgPipeline.from_pretrained(
    'runwayml/stable-diffusion-v1-5',
    torch_dtype=dtype,
    safety_checker=None
).to(device)

STRENGTH  = 0.45   # low strength preserves the TD composition
BB_WEIGHT = 1.2   # boost backbone brightness before blend (black bg + thin lines)
results   = {}

for gene, prompt in prompts.items():
    base_path = f'images/img2img_{gene}.png'
    bb_gene = BB_FALLBACK.get(gene, gene)
    bb_path = f'images/structures/{bb_gene}_structure.png'

    if not os.path.exists(base_path):
        print(f'  [{gene}] Step 8 output not found, skipping')
        continue
    if not os.path.exists(bb_path):
        print(f'  [{gene}] backbone not found, skipping')
        continue

    base = Image.open(base_path).convert('RGB').resize((512, 512))
    bb   = Image.open(bb_path).convert('RGB').resize((512, 512))

    # Boost backbone so thin lines glow visibly, then screen blend onto base
    bb_arr    = np.clip(np.array(bb).astype(float) * BB_WEIGHT, 0, 255).astype(np.uint8)
    bb_bright = Image.fromarray(bb_arr)
    blended   = screen_blend(base, bb_bright, weight=1.0)

    print(f'  [{gene}] img2img strength={STRENGTH}...')
    generator = torch.Generator(device=device).manual_seed(42)
    result = pipe(
        prompt=prompt,
        negative_prompt='blurry, flat, matte, white background, cartoon, dull',
        image=blended,
        strength=STRENGTH,
        num_inference_steps=30,
        guidance_scale=8.0,
        generator=generator
    ).images[0]

    out_path = f'images/8c/{gene}_8c.png'
    result.save(out_path)
    results[gene] = result
    print(f'  Saved -> {out_path}')

print(f'\nDone: {len(results)} images saved to images/8c/')

n    = len(results)
cols = 4
rows = (n + cols - 1) // cols
fig, axes = plt.subplots(rows, cols, figsize=(cols*4, rows*4), facecolor='black')
axes = axes.flatten()
for i, (gene, img) in enumerate(results.items()):
    axes[i].imshow(img)
    axes[i].set_title(gene, fontsize=10, color='lime' if gene == 'luxZ' else 'white')
    axes[i].axis('off')
for j in range(i+1, len(axes)):
    axes[j].axis('off')
plt.suptitle('Step 8c — TD img2img + backbone glow (strength=0.3)', color='white', fontsize=13)
plt.tight_layout()
plt.savefig('images/8c/all_8c.png', dpi=150, facecolor='black')
plt.show()
print('Saved -> images/8c/all_8c.png')


## Step 8c-2 — ControlNet Backbone + TD img2img

**Why this exists**: Step 8c produced a simple overlay — the backbone lines sat on top of the TD composition without reshaping it. ControlNet fixes this by feeding the backbone geometry directly into the SD generation process, so the protein structure actively influences how the image is redrawn rather than just being blended on top.

**Pipeline**:
```
Step 8 output + backbone (inverted) → ControlNet scribble + img2img → images/8c2/{gene}_8c2.png
```

**Note**: Backbone is inverted before ControlNet — scribble conditioning expects white background with dark lines.

**Parameters**: `strength=0.65`, `guidance_scale=3.0`, `controlnet_conditioning_scale=2.5`

In [ ]:
# Backbone fallback: genes that share a structure with another gene
BB_FALLBACK = {'luxZ_water': 'luxZ'}

import torch, os, json, numpy as np, matplotlib.pyplot as plt
from PIL import Image, ImageOps
from diffusers import StableDiffusionControlNetImg2ImgPipeline, ControlNetModel, UniPCMultistepScheduler

os.makedirs('images/8c2', exist_ok=True)

with open('data/sd_prompts.json') as f:
    prompts = json.load(f)

if torch.cuda.is_available():
    device, dtype = 'cuda', torch.float16
elif torch.backends.mps.is_available():
    device, dtype = 'mps', torch.float32
else:
    device, dtype = 'cpu', torch.float32

print('Loading ControlNet scribble...')
controlnet = ControlNetModel.from_pretrained(
    'lllyasviel/control_v11p_sd15_scribble',
    torch_dtype=dtype
).to(device)

print('Loading SD img2img + ControlNet pipeline...')
pipe = StableDiffusionControlNetImg2ImgPipeline.from_pretrained(
    'runwayml/stable-diffusion-v1-5',
    controlnet=controlnet,
    torch_dtype=dtype,
    safety_checker=None
).to(device)
pipe.scheduler = UniPCMultistepScheduler.from_config(pipe.scheduler.config)
print('Pipeline ready.')

STRENGTH                    = 0.65 # higher strength = more SD creativity, lower = preserve TD + backbone more
GUIDANCE_SCALE              = 3.0 # higher = follow prompt more, lower = follow image more
CONTROLNET_CONDITIONING_SCALE = 2.5 # higher = follow backbone scribble more, lower = follow img2img base more
results        = {}

for gene, prompt in prompts.items():
    base_path = f'images/img2img_{gene}.png'
    bb_gene = BB_FALLBACK.get(gene, gene)
    bb_path = f'images/structures/{bb_gene}_structure.png'

    if not os.path.exists(base_path):
        print(f'  [{gene}] Step 8 output not found, skipping')
        continue
    if not os.path.exists(bb_path):
        print(f'  [{gene}] backbone trace not found, skipping')
        continue

    base = Image.open(base_path).convert('RGB').resize((512, 512))
    bb   = Image.open(bb_path).convert('RGB').resize((512, 512))

    # Scribble ControlNet expects white background + dark lines
    # Our backbone is black background + coloured lines, so invert
    bb_cond = ImageOps.invert(bb)

    print(f'  [{gene}] ControlNet scribble + img2img (strength={STRENGTH})...')
    generator = torch.Generator(device=device).manual_seed(42)
    result = pipe(
        prompt=prompt,
        negative_prompt='blurry, flat, matte, white background, cartoon, dull',
        image=base,
        control_image=bb_cond,
        strength=STRENGTH,
        num_inference_steps=30,
        guidance_scale=GUIDANCE_SCALE,
        controlnet_conditioning_scale=float(CONTROLNET_CONDITIONING_SCALE),
        generator=generator
    ).images[0]

    out_path = f'images/8c2/{gene}_8c2.png'
    result.save(out_path)
    results[gene] = result
    print(f'  Saved -> {out_path}')

print(f'\nDone: {len(results)} images saved to images/8c2/')

n    = len(results)
cols = 4
rows = (n + cols - 1) // cols
fig, axes = plt.subplots(rows, cols, figsize=(cols*4, rows*4), facecolor='black')
axes = axes.flatten()
for i, (gene, img) in enumerate(results.items()):
    axes[i].imshow(img)
    axes[i].set_title(gene, fontsize=10, color='lime' if gene == 'luxZ' else 'white')
    axes[i].axis('off')
for j in range(i+1, len(axes)):
    axes[j].axis('off')
plt.suptitle('Step 8c-2 — ControlNet scribble backbone + TD img2img (strength=0.5)', color='white', fontsize=13)
plt.tight_layout()
plt.savefig('images/8c2/all_8c2.png', dpi=150, facecolor='black')
plt.show()
print('Saved -> images/8c2/all_8c2.png')


## Step 8d — Lab Blend: 8b Colour × 8c-2 Structure

**Why this exists**: 8b has the organic cell depth and bioluminescent colour we want, but lacks the protein geometry. 8c-2 has the backbone structure but loses the cell feel. Lab colour space lets us separate these two qualities and recombine them selectively.

**How it works**: Both images are converted to Lab colour space, where L controls lightness/depth and ab controls colour. We blend the L channel 50% from 8b and 50% from 8c-2, then keep the ab channels fully from 8b — so the cell colour and bioluminescent atmosphere dominate, while the protein geometry is incorporated into the depth layer.

**No SD involved** — pure mathematical blend, fully deterministic.

In [ ]:
import os, json, numpy as np, matplotlib.pyplot as plt
from PIL import Image
from skimage import color

os.makedirs('images/8d', exist_ok=True)

with open('data/sd_prompts.json') as f:
    prompts = json.load(f)

# luxZ_water shares outputs with luxZ variants
IMG_FALLBACK = {'luxZ_water': 'luxZ'}

results = {}

for gene in prompts:
    base_gene = IMG_FALLBACK.get(gene, gene)

    path_8c2 = f'images/8c2/{gene}_8c2.png'
    path_8b  = f'images/td_composite/{base_gene}_td_composite.png'

    # Try gene-specific 8c2, fall back to base gene
    if not os.path.exists(path_8c2):
        path_8c2 = f'images/8c2/{base_gene}_8c2.png'
    if not os.path.exists(path_8c2):
        print(f'  [{gene}] 8c-2 output not found, skipping')
        continue
    if not os.path.exists(path_8b):
        print(f'  [{gene}] 8b output not found ({path_8b}), skipping')
        continue

    img_8c2 = np.array(Image.open(path_8c2).convert('RGB').resize((512, 512))).astype(np.float64) / 255.0
    img_8b  = np.array(Image.open(path_8b).convert('RGB').resize((512, 512))).astype(np.float64) / 255.0

    # Convert both to Lab
    lab_8c2 = color.rgb2lab(img_8c2)   # geometry / structure
    lab_8b  = color.rgb2lab(img_8b)    # colour source

    # L: 70% from 8b (organic depth) + 30% from 8c-2 (backbone structure)
    # ab: fully from 8b (colour)
    lab_blend = lab_8b.copy()
    lab_blend[:, :, 0] = 0.5 * lab_8b[:, :, 0] + 0.5 * lab_8c2[:, :, 0]
    lab_blend[:, :, 1] = lab_8b[:, :, 1]
    lab_blend[:, :, 2] = lab_8b[:, :, 2]

    rgb_blend = color.lab2rgb(lab_blend)
    result    = Image.fromarray((np.clip(rgb_blend, 0, 1) * 255).astype(np.uint8))

    out_path = f'images/8d/{gene}_8d.png'
    result.save(out_path)
    results[gene] = result
    print(f'  [{gene}] saved -> {out_path}')

print(f'\nDone: {len(results)} images saved to images/8d/')

n    = len(results)
cols = 4
rows = (n + cols - 1) // cols
fig, axes = plt.subplots(rows, cols, figsize=(cols*4, rows*4), facecolor='black')
axes = axes.flatten()
for i, (gene, img) in enumerate(results.items()):
    axes[i].imshow(img)
    axes[i].set_title(gene, fontsize=10, color='lime' if gene == 'luxZ' else 'white')
    axes[i].axis('off')
for j in range(i+1, len(axes)):
    axes[j].axis('off')
plt.suptitle('Step 8d — Lab blend: 8c-2 structure × 8b colour', color='white', fontsize=13)
plt.tight_layout()
plt.savefig('images/8d/all_8d.png', dpi=150, facecolor='black')
plt.show()
print('Saved -> images/8d/all_8d.png')


---
## Step 8e — Video Input × Lux Biology

Same pipeline as Step 8 (img2img per gene) but using `TDMovieOut.0.mov` as input
instead of a single TIF. Each gene produces one output GIF.

**Video**: 1280×720, 60fps, 12.4s → extracted at 5fps = ~62 frames
**Output**: `images/video_out/{gene}_out.gif`


In [ ]:
import os, torch, json, glob, subprocess
from PIL import Image
from diffusers import StableDiffusionImg2ImgPipeline
from IPython.display import Image as IPImage, display

# ----------------------------------------------------------------
VIDEO_PATH       = 'TDMovieOut.0.mov'
FPS              = 5
STRENGTH         = 0.6
INFERENCE_STEPS  = 20
GENES_TO_PROCESS = ['luxA', 'luxB', 'luxI', 'luxR', 'luxZ']
# ----------------------------------------------------------------

# --- 1. Extract frames ---
frames_dir = 'images/video_frames'
os.makedirs(frames_dir, exist_ok=True)
for f in glob.glob(f'{frames_dir}/frame_*.png'):
    os.remove(f)

import shutil
FFMPEG = shutil.which('ffmpeg') or '/usr/bin/ffmpeg'
subprocess.run([
    FFMPEG, '-y', '-i', VIDEO_PATH,
    '-vf', f'fps={FPS},scale=512:512:force_original_aspect_ratio=decrease,pad=512:512:(ow-iw)/2:(oh-ih)/2',
    f'{frames_dir}/frame_%04d.png'
], check=True)

frame_paths = sorted(glob.glob(f'{frames_dir}/frame_*.png'))
print(f'Extracted {len(frame_paths)} frames')
display(Image.open(frame_paths[0]))

# --- 2. Load pipeline (same as Step 8) ---
if torch.cuda.is_available():
    device, dtype = 'cuda', torch.float16
elif torch.backends.mps.is_available():
    device, dtype = 'mps', torch.float32
else:
    device, dtype = 'cpu', torch.float32

pipe = StableDiffusionImg2ImgPipeline.from_pretrained(
    'runwayml/stable-diffusion-v1-5',
    torch_dtype=dtype,
    safety_checker=None
).to(device)

with open('data/sd_prompts.json') as f:
    prompts = {g: p for g, p in json.load(f).items() if g in GENES_TO_PROCESS}

print(f'Genes: {list(prompts.keys())}')
print(f'~{len(frame_paths)*len(prompts)*1.5/60:.0f} min estimated on MPS')

# --- 3. Run img2img per gene (same as Step 8) ---
os.makedirs('images/video_out', exist_ok=True)

for gene, prompt in prompts.items():
    gene_dir = f'images/video_out/{gene}_frames'
    os.makedirs(gene_dir, exist_ok=True)
    print(f'\n[{gene}]')

    for j, frame_path in enumerate(frame_paths):
        out_path = f'{gene_dir}/frame_{j:04d}.png'
        if os.path.exists(out_path):
            continue
        frame     = Image.open(frame_path).convert('RGB')
        generator = torch.Generator(device=device).manual_seed(42)
        result    = pipe(
            prompt=prompt,
            image=frame,
            strength=STRENGTH,
            num_inference_steps=INFERENCE_STEPS,
            guidance_scale=7.5,
            generator=generator
        ).images[0]
        result.save(out_path)
        if (j+1) % 10 == 0:
            print(f'  {j+1}/{len(frame_paths)}')

    # Assemble GIF
    out_frames = [Image.open(p) for p in sorted(glob.glob(f'{gene_dir}/frame_*.png'))]
    gif_path   = f'images/video_out/{gene}_out.gif'
    out_frames[0].save(
        gif_path,
        save_all=True,
        append_images=out_frames[1:],
        duration=int(1000/FPS),
        loop=0
    )
    print(f'  -> {gif_path}')
    display(IPImage(gif_path))

print('\nAll done. GIFs saved to images/video_out/')


## Step 9 — Organism Video

Takes the per-gene video frames from Step 8e and screen-blends them together frame-by-frame using biological weights.

**Biological weights**: luxA and luxB at full intensity (direct light emitters), luxZ at 0.85 (synthetic hybrid), luxI and luxR at 0.5 (regulators).

**Input**: `images/video_out/{gene}_frames/` (from Step 8e)
**Output**: `images/video_out/organism_video.gif`

In [ ]:
import os, glob, numpy as np
from PIL import Image
from IPython.display import Image as IPImage, display

# Biological weights — same as Step 9
GENE_WEIGHTS = {
    'luxA': 1.00,
    'luxB': 1.00,
    'luxZ': 0.85,
    'luxI': 0.50,
    'luxR': 0.50,
}

def screen_blend(base, layer, weight=1.0):
    a = np.array(base).astype(float) / 255
    b = np.array(layer).astype(float) / 255 * weight
    return Image.fromarray((np.clip(1-(1-a)*(1-b), 0, 1)*255).astype(np.uint8))

# --- Load frame paths per gene ---
gene_frames = {}
for gene in GENE_WEIGHTS:
    paths = sorted(glob.glob(f'images/video_out/{gene}_frames/frame_*.png'))
    if paths:
        gene_frames[gene] = paths
        print(f'{gene:6s}: {len(paths)} frames')
    else:
        print(f'{gene:6s}: NOT FOUND — run Step 8c first')

if not gene_frames:
    raise RuntimeError('No gene frames found. Run Step 8c first.')

n_frames = min(len(p) for p in gene_frames.values())
print(f'\nBlending {n_frames} frames across {len(gene_frames)} genes...')

# --- Blend frame by frame ---
os.makedirs('images/video_out/organism_frames', exist_ok=True)
blended_frames = []

for j in range(n_frames):
    canvas = Image.new('RGB', (512, 512), 0)
    for gene, weight in GENE_WEIGHTS.items():
        if gene not in gene_frames:
            continue
        frame  = Image.open(gene_frames[gene][j]).convert('RGB')
        canvas = screen_blend(canvas, frame, weight=weight)

    out_path = f'images/video_out/organism_frames/frame_{j:04d}.png'
    canvas.save(out_path)
    blended_frames.append(canvas)
    if (j+1) % 10 == 0:
        print(f'  {j+1}/{n_frames}')

print(f'All {n_frames} frames blended.')

# --- Assemble GIF ---
gif_path = 'images/video_out/organism_video.gif'
blended_frames[0].save(
    gif_path,
    save_all=True,
    append_images=blended_frames[1:],
    duration=200,
    loop=0
)
print(f'Saved -> {gif_path}')
display(IPImage(gif_path))


## Step 9b — Gene Identity Morph

Takes all 11 gene images from step 8d and morphs between them sequentially, ordered by lux box score from high to low (luxI → luxR → ... → luxD → back to luxI). 

**Input**: `images/8d/{gene}_8d.png`
**Output**: `images/organism_morph.gif`

In [ ]:
import os
import numpy as np
from PIL import Image
from IPython.display import Image as IPImage, display

# Genes ordered by lux box score, high → low
GENE_ORDER = ['luxI', 'luxR', 'qsrP', 'luxZ', 'luxE', 'luxC', 'luxT', 'luxG', 'luxB', 'luxA', 'luxD']

SIZE    = 512
N_TRANS = 15      # linear-blend frames per adjacent pair
FPS     = 12
OUT_GIF = 'images/organism_morph.gif'

gene_imgs = {}
for gene in GENE_ORDER:
    path = f'images/8d/{gene}_8d.png'
    if not os.path.exists(path):
        print(f'  [{gene}] 8d image not found, skipping')
        continue
    gene_imgs[gene] = (
        np.array(Image.open(path).convert('RGB').resize((SIZE, SIZE))).astype(np.float64) / 255.0
    )

loaded = [g for g in GENE_ORDER if g in gene_imgs]
print(f'Loaded {len(loaded)} gene images: {loaded}')

frames = []
for i in range(len(loaded) - 1):
    gene_a, gene_b = loaded[i], loaded[i + 1]
    img_a, img_b   = gene_imgs[gene_a], gene_imgs[gene_b]
    for t in np.linspace(0.0, 1.0, N_TRANS, endpoint=False):
        blended = (1.0 - t) * img_a + t * img_b
        frames.append(Image.fromarray((np.clip(blended, 0, 1) * 255).astype(np.uint8)))
    print(f'  {gene_a} -> {gene_b}: {N_TRANS} frames')

# Loop back: luxD → luxI (closes the animation cycle)
gene_a, gene_b = loaded[-1], loaded[0]
img_a, img_b   = gene_imgs[gene_a], gene_imgs[gene_b]
for t in np.linspace(0.0, 1.0, N_TRANS, endpoint=False):
    blended = (1.0 - t) * img_a + t * img_b
    frames.append(Image.fromarray((np.clip(blended, 0, 1) * 255).astype(np.uint8)))

# Hold on the last gene for one frame
# frames.append(Image.fromarray((np.clip(gene_imgs[loaded[-1]], 0, 1) * 255).astype(np.uint8)))

frames[0].save(
    OUT_GIF,
    save_all=True,
    append_images=frames[1:],
    duration=int(1000 / FPS),
    loop=0
)
print(f'\nSaved -> {OUT_GIF}  ({len(frames)} frames @ {FPS} fps)')

display(IPImage(OUT_GIF))

---
## Step 9c — Comparison: Organism vs Gene Identity

**What this does**: Combines Step 9b (screen-blended organism video) and Step 9b (gene morph) side by side.

- **Left panel** — organism in its environment (Step 9b frames from `images/video_out/organism_frames/`)
- **Right panel** — gene identity morphing sequence (Step 9b logic, frames saved to `images/organism_morph_frames/`)
- Both panels are 512×512; the combined canvas is 1024×534 (512 + 22px label strip)
- If frame counts differ, the shorter sequence loops to match the longer
- Output: `images/organism_dualview.gif` at 12 fps

---

In [ ]:
import os, glob
import numpy as np
from PIL import Image, ImageDraw, ImageFont
from IPython.display import Image as IPImage, display

# ── Left panel: Step 9 organism frames ───────────────────────────────────────
left_paths = sorted(glob.glob('images/video_out/organism_frames/frame_*.png'))
if not left_paths:
    raise RuntimeError('No organism frames found. Run Step 9 first.')
print(f'Left  (9b): {len(left_paths)} frames')

# ── Right panel: regenerate Step 9b morph frames ──────────────────────────────
GENE_ORDER = ['luxI', 'luxR', 'qsrP', 'luxZ', 'luxE', 'luxC', 'luxT', 'luxG', 'luxB', 'luxA', 'luxD']
SIZE    = 512
N_TRANS = 15

gene_imgs = {}
for gene in GENE_ORDER:
    path = f'images/8d/{gene}_8d.png'
    if not os.path.exists(path):
        print(f'  [{gene}] 8d image not found, skipping')
        continue
    gene_imgs[gene] = (
        np.array(Image.open(path).convert('RGB').resize((SIZE, SIZE))).astype(np.float64) / 255.0
    )

loaded = [g for g in GENE_ORDER if g in gene_imgs]
print(f'Right (9c): {len(loaded)} genes loaded')

os.makedirs('images/organism_morph_frames', exist_ok=True)

morph_paths = []
frame_idx = 0
for i in range(len(loaded) - 1):
    img_a, img_b = gene_imgs[loaded[i]], gene_imgs[loaded[i + 1]]
    for t in np.linspace(0.0, 1.0, N_TRANS, endpoint=False):
        blended = (1.0 - t) * img_a + t * img_b
        p = f'images/organism_morph_frames/frame_{frame_idx:04d}.png'
        Image.fromarray((np.clip(blended, 0, 1) * 255).astype(np.uint8)).save(p)
        morph_paths.append(p)
        frame_idx += 1

last = Image.fromarray((np.clip(gene_imgs[loaded[-1]], 0, 1) * 255).astype(np.uint8))
p = f'images/organism_morph_frames/frame_{frame_idx:04d}.png'
last.save(p)
morph_paths.append(p)
print(f'Right (9c): {len(morph_paths)} morph frames saved -> images/organism_morph_frames/')

# ── Compose dual-view frames ───────────────────────────────────────────────────
LABEL_H = 22
DUAL_W  = SIZE * 2       # 1024
DUAL_H  = SIZE + LABEL_H # 534

font = ImageFont.load_default()

def center_x(text, panel_x, panel_w):
    bbox = font.getbbox(text)
    return panel_x + (panel_w - (bbox[2] - bbox[0])) // 2

n_left   = len(left_paths)
n_right  = len(morph_paths)
n_frames = max(n_left, n_right)
print(f'\nCompositing {n_frames} dual-view frames (left={n_left}, right={n_right})...')

OUT_GIF = 'images/organism_dualview.gif'
FPS     = 12
frames  = []

for j in range(n_frames):
    left  = Image.open(left_paths[j % n_left]).convert('RGB').resize((SIZE, SIZE))
    right = Image.open(morph_paths[j % n_right]).convert('RGB').resize((SIZE, SIZE))

    canvas = Image.new('RGB', (DUAL_W, DUAL_H), (0, 0, 0))
    canvas.paste(left,  (0,    0))
    canvas.paste(right, (SIZE, 0))

    draw = ImageDraw.Draw(canvas)
    y = SIZE + 5
    draw.text((center_x('organism in environment', 0,    SIZE), y), 'organism in environment', fill=(200, 200, 200), font=font)
    draw.text((center_x('gene identity',           SIZE, SIZE), y), 'gene identity',           fill=(200, 200, 200), font=font)

    frames.append(canvas)
    if (j + 1) % 20 == 0:
        print(f'  {j+1}/{n_frames}')

frames[0].save(
    OUT_GIF,
    save_all=True,
    append_images=frames[1:],
    duration=int(1000 / FPS),
    loop=0
)
print(f'\nSaved -> {OUT_GIF}  ({n_frames} frames @ {FPS} fps, {DUAL_W}\u00d7{DUAL_H})')

display(IPImage(OUT_GIF))

## Step 10 — Animation: luxI Emits the Signal

While Steps 9b and 9c assemble the organism from existing 8d images, Step 10 takes a different approach: it generates new frames from scratch using chained img2img, letting SD narrate the biological story rather than compositing pre-existing outputs.

**Starting from the TouchDesigner input image**, each of the 16 frames is generated by passing the previous frame through img2img with a prompt describing the current biological state — tracing the full activation arc from a dark dormant cell through to maximum bioluminescence and back.

```
TD image → dark cell → luxI fires → AHL builds → lux box ON → luxA/B erupt → sustained glow → loop
```

Each frame evolves continuously from the previous via chained img2img. A bloom effect is applied post-generation to amplify the bioluminescent glow. The 16 frames are assembled forward then reversed into a seamless loop.

**Input**: `TDimage1.tif`  
**Output**: `images/lux_emission.gif` (32 frames at 120ms/frame, ~3.8s loop)

In [ ]:
import json
import torch
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from diffusers import StableDiffusionImg2ImgPipeline
import os

# The biological story: 16 frames from dark → full bioluminescence
FRAMES = [
    # (quorum_state, prompt_description, strength)
    (0.00, "dormant Vibrio fischeri bacterium, dark cell, no light, inactive", 0.55),
    (0.05, "dormant bacterium, very faint metabolic activity, nearly dark", 0.50),
    (0.10, "luxI gene activating, single molecule of autoinducer produced, trace glow", 0.50),
    (0.20, "luxI producing autoinducer signal, faint amber flicker, quorum sensing beginning", 0.50),
    (0.30, "autoinducer molecules accumulating, dim warm glow, luxR sensing the signal", 0.50),
    (0.40, "autoinducer reaching threshold, luxR binding, lux box about to switch on, soft glow", 0.50),
    (0.50, "lux box activated by LuxR, lux operon switching on, blue-green glow emerging", 0.50),
    (0.60, "luxC luxD luxE producing aldehyde fuel, growing blue-green bioluminescence", 0.48),
    (0.70, "luciferase reaction beginning, luxA luxB activating, bright blue-green light", 0.48),
    (0.80, "luciferase fully active, intense blue-green bioluminescent glow, Vibrio fischeri", 0.45),
    (0.90, "maximum bioluminescence, brilliant blue-green light, full lux operon active", 0.45),
    (0.95, "blazing blue-green bioluminescence, entire lux system at peak, LuxZ contributing", 0.42),
    (1.00, "full Vibrio fischeri bioluminescence, brilliant blue-green light, dark ocean, 8k", 0.40),
    (0.95, "blazing bioluminescence pulsing, quorum sensing maintained, glowing colony", 0.40),
    (0.85, "sustained bioluminescence, steady blue-green glow, stable quorum", 0.42),
    (0.70, "bioluminescence continuing, lux genes active, blue-green light, dark ocean", 0.45),
]

BASE_PROMPT_SUFFIX = (
    ", bioluminescent bacterium, dark deep ocean background, "
    "scientific macro photography, photorealistic, cinematic lighting, 8k"
)
NEGATIVE_PROMPT = "bright daylight, daytime, white background, blurry, cartoon, illustration"

# Load the organism composite from Step 9 as the starting frame
# Fall back to TD image if Step 9 wasn't run
start_path = 'TDimage1.tif'


start_img = Image.open(start_path).convert('RGB').resize((512, 512))
print(f'Starting frame loaded from: {start_path}')

# Device
if torch.cuda.is_available():
    device, dtype = 'cuda', torch.float16
elif torch.backends.mps.is_available():
    device, dtype = 'mps', torch.float32
else:
    device, dtype = 'cpu', torch.float32

pipe = StableDiffusionImg2ImgPipeline.from_pretrained(
    'runwayml/stable-diffusion-v1-5',
    torch_dtype=dtype,
    safety_checker=None
).to(device)

os.makedirs('images/frames', exist_ok=True)

frames = []
current_img = start_img  # each frame feeds into the next

print(f'\nGenerating {len(FRAMES)} frames (chained img2img):')
for i, (qs_state, description, strength) in enumerate(FRAMES):
    prompt = description + BASE_PROMPT_SUFFIX
    print(f'  Frame {i:02d} | QS={qs_state:.2f} | {description[:50]}...')

    generator = torch.Generator(device=device).manual_seed(42 + i)
    result = pipe(
        prompt=prompt,
        negative_prompt=NEGATIVE_PROMPT,
        image=current_img,
        strength=strength,
        num_inference_steps=35,
        generator=generator
    )
    frame = result.images[0]
    frame_path = f'images/frames/frame_{i:02d}.png'
    frame.save(frame_path)
    frames.append(frame)
    current_img = frame  # chain: next frame starts from this one

print(f'\nAll {len(frames)} frames generated.')

# Assemble GIF
# Forward then reverse for a smooth loop
loop_frames = frames + frames[::-1]

gif_path = 'images/lux_emission.gif'
loop_frames[0].save(
    gif_path,
    save_all=True,
    append_images=loop_frames[1:],
    duration=120,   # ms per frame
    loop=0          # loop forever
)
print(f'GIF saved -> {gif_path}')
print(f'  {len(loop_frames)} frames total, 120ms each = ~{len(loop_frames)*120/1000:.1f}s loop')

# Show all frames in a grid
cols = 8
rows = (len(frames) + cols - 1) // cols
fig, axes = plt.subplots(rows, cols, figsize=(cols * 2.5, rows * 2.5))
axes = axes.flatten()

for i, frame in enumerate(frames):
    qs = FRAMES[i][0]
    axes[i].imshow(frame)
    axes[i].set_title(f'f{i} QS={qs:.2f}', fontsize=7, color='white')
    axes[i].axis('off')

for j in range(len(frames), len(axes)):
    axes[j].axis('off')

fig.patch.set_facecolor('black')
plt.suptitle('luxI Emission Animation — Frame Preview', color='white', fontsize=12)
plt.tight_layout()
plt.savefig('images/frames_preview.png', dpi=120, facecolor='black')
plt.show()
print('Frame preview saved -> images/frames_preview.png')

In [ ]:
import IPython
IPython.display.Image('images/lux_emission.gif')

In [ ]:
#Add Bloom / Glow Effect to Existing Frames
# Applies a bloom effect by screen-blending a blurred version of the bright pixels
# back on top — no re-generation needed.

from PIL import Image, ImageFilter
import numpy as np
import glob

def add_bloom(img, blur_radius=18, intensity=1.4, threshold=60):
    arr = np.array(img).astype(float)
    # isolate bright pixels and amplify them
    bright = np.clip(arr - threshold, 0, 255) * intensity
    bloom = Image.fromarray(bright.astype(np.uint8)).filter(
        ImageFilter.GaussianBlur(radius=blur_radius)
    )
    # screen blend: light from bright regions spreads outward
    a = arr / 255
    b = np.array(bloom) / 255
    result = 1 - (1 - a) * (1 - b)
    return Image.fromarray((np.clip(result, 0, 1) * 255).astype(np.uint8))

frame_paths = sorted(glob.glob('images/frames/frame_*.png'))
print(f'Found {len(frame_paths)} frames')

glow_frames = [add_bloom(Image.open(p).convert('RGB')) for p in frame_paths]

loop_frames = glow_frames + glow_frames[::-1]
out_path = 'images/lux_emission_glow.gif'
glow_frames[0].save(
    out_path,
    save_all=True,
    append_images=loop_frames[1:],
    duration=120,
    loop=0
)
print(f'Saved -> {out_path}')

from IPython.display import Image as IPImage, display
display(IPImage(out_path))

## Step 10b — Bioluminescence Activation: luxI Start

A second version of the Step 10 animation, starting from `images/8d/luxI_8d.png` instead of a blank organism composite. This connects directly to Step 9b: the gene identity morph sequence arrives at luxI — the gene with the highest quorum sensing score — and this animation picks up from that exact visual state.

The same 16-frame chained img2img logic applies: each frame evolves from the previous using a prompt that describes the current biological state, tracing the full activation arc from a dark dormant cell through to maximum bioluminescence and back.

```
luxI 8d image → dark cell → AHL builds → lux box ON → luxA/B erupt → sustained glow → loop
```

Bloom effect applied post-generation. Output is a seamless forward-reverse loop.

**Input**: `images/8d/luxI_8d.png`  
**Output**: `images/lux_emission_luxZ_start.gif`

In [ ]:
import torch, os
import numpy as np
from PIL import Image, ImageFilter
from diffusers import StableDiffusionImg2ImgPipeline
from IPython.display import Image as IPImage, display

# ── Same 16-frame biological story as Step 10 ─────────────────────────────────
FRAMES = [
    (0.00, "dormant Vibrio fischeri bacterium, dark cell, no light, inactive", 0.55),
    (0.05, "dormant bacterium, very faint metabolic activity, nearly dark", 0.50),
    (0.10, "luxI gene activating, single molecule of autoinducer produced, trace glow", 0.50),
    (0.20, "luxI producing autoinducer signal, faint amber flicker, quorum sensing beginning", 0.50),
    (0.30, "autoinducer molecules accumulating, dim warm glow, luxR sensing the signal", 0.50),
    (0.40, "autoinducer reaching threshold, luxR binding, lux box about to switch on, soft glow", 0.50),
    (0.50, "lux box activated by LuxR, lux operon switching on, blue-green glow emerging", 0.50),
    (0.60, "luxC luxD luxE producing aldehyde fuel, growing blue-green bioluminescence", 0.48),
    (0.70, "luciferase reaction beginning, luxA luxB activating, bright blue-green light", 0.48),
    (0.80, "luciferase fully active, intense blue-green bioluminescent glow, Vibrio fischeri", 0.45),
    (0.90, "maximum bioluminescence, brilliant blue-green light, full lux operon active", 0.45),
    (0.95, "blazing blue-green bioluminescence, entire lux system at peak, LuxZ contributing", 0.42),
    (1.00, "full Vibrio fischeri bioluminescence, brilliant blue-green light, dark ocean, 8k", 0.40),
    (0.95, "blazing bioluminescence pulsing, quorum sensing maintained, glowing colony", 0.40),
    (0.85, "sustained bioluminescence, steady blue-green glow, stable quorum", 0.42),
    (0.70, "bioluminescence continuing, lux genes active, blue-green light, dark ocean", 0.45),
]

BASE_PROMPT_SUFFIX = (
    ", bioluminescent bacterium, dark deep ocean background, "
    "scientific macro photography, photorealistic, cinematic lighting, 8k"
)
NEGATIVE_PROMPT = "bright daylight, daytime, white background, blurry, cartoon, illustration"

FRAME_DIR = 'images/frames_luxI_start'
os.makedirs(FRAME_DIR, exist_ok=True)

# ── Reuse pipe from memory if already loaded, otherwise load fresh ─────────────
if torch.cuda.is_available():
    device, dtype = 'cuda', torch.float16
elif torch.backends.mps.is_available():
    device, dtype = 'mps', torch.float32
else:
    device, dtype = 'cpu', torch.float32

if 'pipe' not in dir() or not isinstance(pipe, StableDiffusionImg2ImgPipeline):
    print('Loading SD pipeline...')
    pipe = StableDiffusionImg2ImgPipeline.from_pretrained(
        'runwayml/stable-diffusion-v1-5',
        torch_dtype=dtype,
        safety_checker=None
    ).to(device)
    print('Pipeline ready.')
else:
    print('Reusing pipe already in memory.')

# ── Load frames from disk if all exist, otherwise generate ────────────────────
frame_paths = [f'{FRAME_DIR}/frame_{i:02d}.png' for i in range(len(FRAMES))]
all_cached  = all(os.path.exists(p) for p in frame_paths)

if all_cached:
    print(f'Loading {len(FRAMES)} cached frames from {FRAME_DIR}/')
    frames = [Image.open(p).convert('RGB') for p in frame_paths]
else:
    start_img = Image.open('images/8d/luxI_8d.png').convert('RGB').resize((512, 512))
    print('Starting frame: images/8d/luxI_8d.png')

    frames      = []
    current_img = start_img

    print(f'Generating {len(FRAMES)} frames (chained img2img):')
    for i, (qs_state, description, strength) in enumerate(FRAMES):
        prompt = description + BASE_PROMPT_SUFFIX
        print(f'  Frame {i:02d} | QS={qs_state:.2f} | {description[:50]}...')
        generator = torch.Generator(device=device).manual_seed(42 + i)
        result = pipe(
            prompt=prompt,
            negative_prompt=NEGATIVE_PROMPT,
            image=current_img,
            strength=strength,
            num_inference_steps=35,
            generator=generator
        )
        frame = result.images[0]
        frame.save(frame_paths[i])
        frames.append(frame)
        current_img = frame

    print(f'All {len(frames)} frames generated and saved to {FRAME_DIR}/')

# ── Bloom effect (same params as Step 11) ─────────────────────────────────────
def add_bloom(img, blur_radius=18, intensity=1.4, threshold=60):
    arr = np.array(img).astype(float)
    bright = np.clip(arr - threshold, 0, 255) * intensity
    bloom = Image.fromarray(bright.astype(np.uint8)).filter(
        ImageFilter.GaussianBlur(radius=blur_radius)
    )
    a = arr / 255
    b = np.array(bloom) / 255
    return Image.fromarray((np.clip(1 - (1 - a) * (1 - b), 0, 1) * 255).astype(np.uint8))

glow_frames = [add_bloom(f) for f in frames]
loop_frames = glow_frames + glow_frames[::-1]

gif_path = 'images/lux_emission_luxZ_start.gif'
loop_frames[0].save(
    gif_path,
    save_all=True,
    append_images=loop_frames[1:],
    duration=120,
    loop=0
)
print(f'Saved -> {gif_path}')
print(f'  {len(loop_frames)} frames total, 120ms each = ~{len(loop_frames)*120/1000:.1f}s loop')

display(IPImage(gif_path))
